# CFPB Narrative Cleaning and Linguistic EDA (v4)

V4 upgrades the validated V3 corpus-readiness notebook into a FinDisputeEval-oriented bridge: parser-ready linguistic features, structured scenario seed candidates, phenomenon-stratified stress sampling, register audit scaffolding, and synthetic-data parameter extraction.

**V4 additions**:
1. Adds an optional spaCy parser hook. If no parser model is installed, the notebook records a deterministic fallback status instead of pretending regex output is parser-backed.
2. Extracts negation scope/focus, passive-event, repair, temporal-role, speech-act layer, narrative-stage order, language-variation, and redaction-type candidate features.
3. Converts narratives into `scenario_seed_candidate` records with claim type, payment rail, route hint, missing slots, risk flags, and human-review reason.
4. Adds phenomenon-stratified stress sampling in addition to the existing register-stratified annotation sample.
5. Adds synthetic generation parameter JSON so the CFPB corpus can drive controlled synthetic dialogue generation and alignment checks.
6. Adds register audit sample scaffolding for validating the rule-based register classifier.
7. Adds collocation output and keyness scatter diagnostics to complement TF-IDF.
8. Splits derived outputs into a local full Parquet file and a repo-safe CSV without raw narratives.
9. Keeps raw CFPB narratives local; repo-safe artifacts contain hashes and derived labels only.
10. All V4 artifacts use versioned filenames and do not overwrite V3 outputs.


## 0. Provenance and data lineage

Every derived artifact in this notebook must be traceable to one specific corpus snapshot. Fill in `CORPUS_METADATA` when the snapshot changes; the SHA-256 is computed from the file itself.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import sys
import unicodedata
from datetime import datetime, timezone
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.metrics import cohen_kappa_score, precision_recall_fscore_support

plt.rcParams["font.family"] = ["Microsoft JhengHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_colwidth", 180)

NOTEBOOK_VERSION = "v4.0"


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the FinDisputeEval project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
CSV_PATH = PROJECT_ROOT / "dataset" / "external" / "cfpb" / "raw" / "ui_export_2026-06-09" / "complaints-2026-06-09_00_45.csv"
RUN_ID = globals().get("RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "data_pipeline" / "cfpb_linguistic_eda" / "eda_v04" / f"run_{RUN_ID}_local"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CORPUS_METADATA = {
    "source": "CFPB Consumer Complaint Database (public export)",
    "source_url": "https://www.consumerfinance.gov/data-research/consumer-complaints/",
    "retrieved": "2026-06-09",
    "export_filters": "Local CFPB export; all 11,817 rows contain a published narrative. "
                      "Exact UI/API filters were not retained with the source file.",
}

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Corpus snapshot not found: {CSV_PATH}\n"
        "Check the external CFPB snapshot path configured above."
    )

corpus_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()
CORPUS_METADATA.update({
    "filename": CSV_PATH.name,
    "sha256": corpus_sha256,
    "size_bytes": CSV_PATH.stat().st_size,
    "notebook_version": NOTEBOOK_VERSION,
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
})

print(json.dumps(CORPUS_METADATA, indent=2))


## 1. Load the corpus

`ZIP code` and `Complaint ID` are identifiers, so they are loaded as strings.


In [ ]:
REQUIRED_COLUMNS = {
    "Complaint ID", "Date received", "Product", "Sub-product", "Issue", "Sub-issue",
    "Consumer complaint narrative", "Company", "State", "ZIP code",
    "Date sent to company", "Timely response?",
}

df = pd.read_csv(
    CSV_PATH,
    dtype={"ZIP code": "string", "Complaint ID": "string"},
    encoding="utf-8-sig",
    low_memory=False,
)

missing_columns = sorted(REQUIRED_COLUMNS - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required CSV columns: {missing_columns}")

for column in ["Date received", "Date sent to company"]:
    df[column] = pd.to_datetime(df[column], errors="coerce", utc=True)

narrative_column = "Consumer complaint narrative"
df["narrative_raw"] = df[narrative_column].fillna("").astype("string")

if df["Complaint ID"].isna().any():
    raise ValueError("Complaint ID contains missing values.")
if not df["Complaint ID"].is_unique:
    raise ValueError("Complaint ID is not unique in this export.")

CORPUS_METADATA["rows"] = int(len(df))
CORPUS_METADATA["rows_with_narrative"] = int((df["narrative_raw"].str.len() > 0).sum())

print(f"Rows: {len(df):,}")
print(f"Rows with non-empty narrative: {CORPUS_METADATA['rows_with_narrative']:,}")
print(f"Columns: {df.shape[1]}")
print(f"Unparseable Date received values: {df['Date received'].isna().sum():,}")
print(f"Unique Complaint IDs: {df['Complaint ID'].nunique():,}")
df[["Complaint ID", "Product", "Issue", "narrative_raw"]].head(3)


## 2. Build non-destructive text views

- `narrative_raw`: untouched source text.
- `narrative_core`: removes known form metadata only when a `Complaint narrative:` section exists.
- `narrative_normalized`: standardizes CFPB placeholders and whitespace for corpus analysis.

Placeholder mapping is intentionally reversible in spirit: `[AMOUNT]`, `[DATE]`, `[REDACTED]` mark *where the censoring happened*, so downstream slot-realization analysis can distinguish "slot present but masked" from "slot absent".


In [ ]:
AMOUNT_RE = re.compile(r"\{\s*\$\s*[\d,.]+\s*\}")
MASKED_DATE_RE = re.compile(r"X{2}/X{2}/(?:X{2,8}|\d{2,4}|year>)", re.I)
REDACTION_RE = re.compile(r"\bX{2,}\b")  # case-sensitive: CFPB redactions are uppercase
CORE_SECTION_RE = re.compile(r"(?is)\bcomplaint narrative\s*:\s*(.+)")


def extract_core_narrative(text: str) -> str:
    text = str(text)
    match = CORE_SECTION_RE.search(text)
    return match.group(1).strip() if match else text.strip()


def normalize_for_analysis(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text))
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = AMOUNT_RE.sub("[AMOUNT]", text)
    text = MASKED_DATE_RE.sub("[DATE]", text)
    text = REDACTION_RE.sub("[REDACTED]", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def strip_placeholders(text: str) -> str:
    """Analysis text with placeholders removed — for measures that placeholders would distort."""
    return re.sub(r"\[(?:REDACTED|DATE|AMOUNT)\]", " ", str(text))


df["narrative_core"] = df["narrative_raw"].map(extract_core_narrative)
df["narrative_normalized"] = df["narrative_core"].map(normalize_for_analysis)
df["narrative_noplaceholder"] = df["narrative_normalized"].map(strip_placeholders)

df[["narrative_raw", "narrative_core", "narrative_normalized"]].head(2)


## 3. Data-quality flags

These flags should be used for stratification and sensitivity analysis. A flag does not automatically mean that a row should be deleted.

`uppercase_ratio` is computed after removing `XXXX`-style redactions. Otherwise, a heavily redacted but lowercase narrative can be misclassified as `mostly_uppercase`, corrupting the written-prosody signal this flag is intended to capture.


In [ ]:
LEGAL_TERM_RE = re.compile(
    r"\b(?:U\.?S\.?C\.?|CFR|FCRA|FDCPA|FCBA|EFTA|Regulation [EZ]|"
    r"statute|statutory|violation|legal action|attorney|lawsuit|pursuant to|"
    r"fair credit reporting act|fair debt collection practices act)\b",
    re.I,
)
FORM_HEADER_RE = re.compile(
    r"(?im)^(?:company|account|consumer|email|address|desired resolution|"
    r"complaint narrative|key facts|timeline|disputed amount|what happened)\s*:"
)
LETTER_MARKER_RE = re.compile(
    r"(?im)^(?:subject|re|dear|to whom it may concern|sincerely|respectfully)\s*[: ,]"
)
ATTACHMENT_RE = re.compile(
    r"\b(?:see attached|attached (?:document|letter|statement|file)|attachment|"
    r"see enclosed|enclosed (?:document|letter|statement))\b",
    re.I,
)

PATTERNS = {
    "has_redaction": re.compile(r"\bX{2,}\b"),
    "has_masked_date": re.compile(r"XX/XX|X{2}/X{2}", re.I),
    "has_masked_amount": re.compile(r"\{\s*\$\s*[\d,.]+\s*\}"),
    "has_template_header": FORM_HEADER_RE,
    "mentions_attachment": ATTACHMENT_RE,
    "letter_or_email_format": LETTER_MARKER_RE,
    "legal_term_present": LEGAL_TERM_RE,
    "contains_url": re.compile(r"https?://|www\.", re.I),
    "repeated_punctuation": re.compile(r"[!?.,-]{4,}"),
}


def uppercase_ratio(text: str) -> float:
    # Remove redaction runs before measuring so XXXX does not count as shouting.
    cleaned = REDACTION_RE.sub(" ", str(text))
    letters = [character for character in cleaned if character.isalpha()]
    if not letters:
        return 0.0
    return sum(character.isupper() for character in letters) / len(letters)


raw = df["narrative_raw"]
for flag, pattern in PATTERNS.items():
    df[flag] = raw.str.contains(pattern, na=False)

WORD_RE = r"\b\w+(?:['’-]\w+)?\b"
df["form_header_count"] = raw.str.count(FORM_HEADER_RE)
df["letter_marker_count"] = raw.str.count(LETTER_MARKER_RE)
df["legal_term_count"] = raw.str.count(LEGAL_TERM_RE)
df["legal_heavy"] = df["legal_term_count"] >= 2

df["char_count"] = df["narrative_core"].str.len()
df["word_count"] = df["narrative_core"].str.findall(WORD_RE).str.len()
df["word_count_noplaceholder"] = df["narrative_noplaceholder"].str.findall(WORD_RE).str.len()
df["newline_count"] = df["narrative_core"].str.count("\n")
df["uppercase_ratio"] = raw.map(uppercase_ratio)
df["redaction_token_count"] = df["narrative_core"].str.count(r"\bX{2,}\b")
df["redaction_density"] = (
    df["redaction_token_count"] / df["word_count"].replace(0, np.nan)
).fillna(0.0)
df["very_short"] = df["char_count"] < 100
df["very_long"] = df["char_count"] > 5000
df["mostly_uppercase"] = df["uppercase_ratio"] > 0.50
df["heavily_redacted"] = df["redaction_density"] > 0.15

df["narrative_hash"] = raw.map(
    lambda text: hashlib.sha1(str(text).encode("utf-8")).hexdigest()
)
df["duplicate_group_size"] = df.groupby("narrative_hash")["narrative_hash"].transform("size")
df["is_exact_duplicate"] = df["duplicate_group_size"] > 1


### 3b. Normalization-equivalent text families

Exact hashing misses template letters where only placeholders and digits differ. The method below removes those elements and hashes the remaining full text. It therefore detects **normalization-equivalent families**. It does not detect paraphrases or general semantic near-duplicates, so V4 no longer calls it near-duplicate detection.

Template families matter for two reasons: (a) they distort prevalence estimates exactly like exact duplicates do; (b) they are their own register — mass-produced advocacy language, not spontaneous consumer narrative.


In [ ]:
def family_signature(text: str) -> str:
    """Hash text after removing placeholders, digits, case, and punctuation."""
    text = strip_placeholders(text).lower()
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return hashlib.sha1(text.encode("utf-8")).hexdigest()


df["family_signature"] = df["narrative_normalized"].map(family_signature)
df["family_group_size"] = (
    df.groupby("family_signature")["family_signature"].transform("size")
)
df["is_normalization_family_member"] = (
    (df["family_group_size"] > 1) & ~df["is_exact_duplicate"]
)

family_only = int(df["is_normalization_family_member"].sum())
print(f"Exact-duplicate rows: {int(df['is_exact_duplicate'].sum()):,}")
print(f"Additional normalization-family rows missed by exact hashing: {family_only:,}")

# Inspect the largest template families before deciding what to do with them.
top_family_groups = (
    df.loc[df["family_group_size"] > 1]
      .groupby("family_signature")
      .agg(group_size=("family_signature", "size"),
           products=("Product", lambda s: s.value_counts().index[0]),
           example=("narrative_normalized", "first"))
      .sort_values("group_size", ascending=False)
      .head(10)
)
top_family_groups["example"] = top_family_groups["example"].str.slice(0, 160)
display(top_family_groups)


In [ ]:
def assign_register(row: pd.Series) -> tuple[str, str, str]:
    evidence = []
    if row["family_group_size"] >= 3:
        evidence.append(f"family_size={int(row['family_group_size'])}")
        return "template_letter_family", "high", "|".join(evidence)
    if row["form_header_count"] >= 2:
        evidence.append(f"form_headers={int(row['form_header_count'])}")
        return "template_form", "high", "|".join(evidence)
    correspondence_score = int(row["letter_marker_count"] >= 1) + int(row["mentions_attachment"])
    if correspondence_score >= 1:
        evidence.append(f"letter_markers={int(row['letter_marker_count'])}")
        evidence.append(f"attachment={bool(row['mentions_attachment'])}")
        confidence = "high" if correspondence_score == 2 else "medium"
        return "pasted_correspondence", confidence, "|".join(evidence)
    if row["legal_term_count"] >= 2:
        evidence.append(f"legal_terms={int(row['legal_term_count'])}")
        confidence = "high" if row["legal_term_count"] >= 4 else "medium"
        return "legal_formal", confidence, "|".join(evidence)
    if row["form_header_count"] == 1 or row["legal_term_count"] == 1:
        evidence.append(f"form_headers={int(row['form_header_count'])}")
        evidence.append(f"legal_terms={int(row['legal_term_count'])}")
        return "consumer_narrative", "low", "|".join(evidence)
    return "consumer_narrative", "medium", "no_structural_marker"


register_labels = df.apply(assign_register, axis=1, result_type="expand")
register_labels.columns = ["register_bucket", "register_confidence", "register_evidence"]
df[["register_bucket", "register_confidence", "register_evidence"]] = register_labels

quality_flags = [
    "has_redaction",
    "has_masked_date",
    "has_masked_amount",
    "has_template_header",
    "mentions_attachment",
    "letter_or_email_format",
    "legal_heavy",
    "very_short",
    "very_long",
    "mostly_uppercase",
    "heavily_redacted",
    "is_exact_duplicate",
    "is_normalization_family_member",
]

quality_summary = pd.DataFrame({
    "count": df[quality_flags].sum(),
    "rate": df[quality_flags].mean(),
}).sort_values("rate", ascending=False)

display(quality_summary.style.format({"rate": "{:.1%}"}))
display(df["register_bucket"].value_counts().to_frame("count"))
display(pd.crosstab(df["register_bucket"], df["register_confidence"]))


## 4. Length and register distributions


In [ ]:
display(df[["char_count", "word_count", "newline_count"]]
        .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).T)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df["word_count"].clip(upper=df["word_count"].quantile(0.99)).plot.hist(
    bins=50, ax=axes[0], title="Narrative word count (capped at p99)"
)
df["register_bucket"].value_counts().plot.bar(
    ax=axes[1], title="Register buckets"
)
axes[0].set_xlabel("Words")
axes[1].set_xlabel("")
plt.tight_layout()
plt.show()


## 5. Create analysis subsets

V3 defines three named views and does not silently switch deduplication policies:
- `all_corpus`: every non-empty row.
- `exact_dedup_corpus`: one row per byte-identical narrative.
- `family_dedup_corpus`: one row per normalization-equivalent family.

Prevalence results are compared across these views. TF-IDF and keyness use `family_dedup_corpus`; annotation sampling starts from `exact_dedup_corpus` and prevents repeated families within the sample.


In [ ]:
all_corpus = df.loc[df["narrative_normalized"].str.len().gt(0)].copy()
exact_dedup_corpus = all_corpus.drop_duplicates("narrative_hash", keep="first").copy()
family_dedup_corpus = exact_dedup_corpus.drop_duplicates(
    "family_signature", keep="first"
).copy()
analysis_corpus = family_dedup_corpus

consumer_voice_corpus = family_dedup_corpus.loc[
    family_dedup_corpus["register_bucket"].eq("consumer_narrative")
].copy()

print(f"All non-empty rows: {len(all_corpus):,}")
print(f"Exact-deduplicated rows: {len(exact_dedup_corpus):,}")
print(f"Family-deduplicated analysis corpus: {len(family_dedup_corpus):,}")
print(f"Consumer-voice sensitivity corpus: {len(consumer_voice_corpus):,}")


## 6. Exploratory linguistic-phenomenon flags and densities

These regex detectors are discovery tools, not gold labels. Review concordance lines (§7), audit pattern precision (§7b), and annotate a sample (§11) before reporting prevalence.

**Use densities, not only booleans.** A document-level boolean for high-frequency phenomena such as negation and deixis saturates: nearly every narrative longer than a few sentences contains at least one occurrence. Per-100-word **density** distinguishes a narrative organized around denial ("I did not authorize... never received... no response...") from one with a single incidental "not". V4 uses a placeholder-free token count as the denominator so redaction markers do not dilute linguistic rates.

**Candidate detectors**, mapped to the FinDisputeEval taxonomy:
- `lx_modality_epistemic` / `lx_modality_deontic` — epistemic hedged certainty ("might have been", "I believe") vs. deontic obligation ("you must refund", "should never have"). The two predict different bot strategies (clarify vs. de-escalate + comply-check).
- `lx_agentless_passive` — restricted to common financial-event participles such as "was closed" and "were removed". It remains a candidate, not a parser-grade passive analysis.
- `lx_caps_emphasis` / expressive punctuation — written prosody; the text-channel analogue of the voice channel's stress and agitation markers.
- `pronoun_density` — replaces the v1 `lx_deixis_candidate` boolean, which matched bare "it" and was therefore true almost everywhere.


In [ ]:
LINGUISTIC_PATTERNS = {
    "lx_negation": re.compile(
        r"\b(?:no|not|never|neither|nor|without|cannot|\w+n['’]t)\b",
        re.I,
    ),
    "lx_hedging": re.compile(
        r"\b(?:I think|I believe|I guess|maybe|might|may|possibly|probably|perhaps|"
        r"not sure|seems?|appears?|pretty sure|as far as I know)\b",
        re.I,
    ),
    "lx_modality_epistemic": re.compile(
        r"\b(?:might have|may have|could have|must have|I (?:believe|think|assume|suspect)|"
        r"probably|possibly|apparently|supposedly|it seems)\b",
        re.I,
    ),
    "lx_modality_deontic": re.compile(
        r"\b(?:you (?:must|need to|have to|should)|they (?:must|need to|have to|should)|"
        r"should (?:not |never )?have|is required to|are required to|obligated to|"
        r"demand that|insist that)\b",
        re.I,
    ),
    "lx_emotion": re.compile(
        r"\b(?:frustrat\w*|ridiculous|angry|upset|stress\w*|distress\w*|"
        r"unacceptable|worst|furious|devastat\w*|scared|afraid)\b",
        re.I,
    ),
    "lx_request_or_demand": re.compile(
        r"\b(?:I want|I need|I request|I ask|please|refund|reverse|remove|correct|"
        r"investigate|resolve|close my account|make this right)\b",
        re.I,
    ),
    "lx_escalation_signal": re.compile(
        r"\b(?:CFPB|regulator|attorney|lawyer|lawsuit|legal action|sue|"
        r"Better Business Bureau|BBB|news media|close my account)\b",
        re.I,
    ),
    "lx_temporal_expression": re.compile(
        r"\b(?:yesterday|today|last (?:week|month|year|night)|this (?:week|month)|"
        r"ago|before|after|when|since|until|recently|on \[DATE\])\b",
        re.I,
    ),
    "lx_authorization_language": re.compile(
        r"\b(?:authori[sz](?:e|ed|ation)|permission|consent|recognize|"
        r"made this (?:charge|purchase|transaction))\b",
        re.I,
    ),
    "lx_self_repair_candidate": re.compile(
        r"\b(?:I mean|actually|rather|correction|no,\s*I mean)\b|\bnot\b.{0,40}\bbut\b",
        re.I,
    ),
    "lx_agentless_passive": re.compile(
        r"\b(?:was|were|is|are|been|being|got)\s+(?:not\s+)?(?:closed|charged|"
        r"denied|removed|reported|stolen|taken|transferred|withdrawn|opened|blocked|"
        r"frozen|cancelled|canceled|rejected|declined|compromised|hacked|misled|"
        r"contacted|told)\b(?!\s+by\b)",
        re.I,
    ),
    "lx_code_switch_candidate": re.compile(
        r"\b(?:no reconozco|mi cuenta|mi tarjeta|por favor|mi dinero|no fui yo|fraude)\b",
        re.I,
    ),
}

# Written-prosody: 3+ letter all-caps tokens that are NOT redactions or common finance acronyms.
CAPS_STOPLIST = {
    "XXXX", "XXX", "CFPB", "FCRA", "FDCPA", "FCBA", "EFTA", "USC", "CFR", "BBB",
    "ACH", "ATM", "APR", "USA", "LLC", "INC", "PDF", "EIN", "SSN", "ID", "VISA",
}
CAPS_TOKEN_RE = re.compile(r"\b[A-Z]{3,}\b")


def caps_emphasis_count(text: str) -> int:
    return sum(
        1 for token in CAPS_TOKEN_RE.findall(str(text))
        if token not in CAPS_STOPLIST and not set(token) == {"X"}
    )


PRONOUN_RE = re.compile(r"\b(?:it|this|that|these|those|they|them)\b", re.I)

# Boolean flags (kept for compatibility and for low-frequency phenomena).
for flag, pattern in LINGUISTIC_PATTERNS.items():
    df[flag] = df["narrative_normalized"].str.contains(pattern, na=False)

# Densities use placeholder-free words so CFPB censoring does not dilute rates.
DENSITY_PHENOMENA = [
    "lx_negation", "lx_hedging", "lx_modality_epistemic", "lx_modality_deontic",
    "lx_emotion", "lx_temporal_expression", "lx_agentless_passive",
]
denominator = df["word_count_noplaceholder"].replace(0, np.nan)
for flag in DENSITY_PHENOMENA:
    df[f"{flag}_density"] = (
        df["narrative_normalized"].str.count(LINGUISTIC_PATTERNS[flag]) / denominator * 100
    ).fillna(0.0)

df["pronoun_density"] = (
    df["narrative_normalized"].str.count(PRONOUN_RE) / denominator * 100
).fillna(0.0)
df["caps_emphasis_count"] = df["narrative_normalized"].map(caps_emphasis_count)
df["expressive_punct_count"] = df["narrative_normalized"].str.count(r"!{2,}|\?{2,}|!\?|\?!")

linguistic_flags = list(LINGUISTIC_PATTERNS)
density_columns = [f"{flag}_density" for flag in DENSITY_PHENOMENA] + ["pronoun_density"]

# Refresh named views so they include the linguistic feature columns created above.
all_corpus = df.loc[df["narrative_normalized"].str.len().gt(0)].copy()
exact_dedup_corpus = all_corpus.drop_duplicates("narrative_hash", keep="first").copy()
family_dedup_corpus = exact_dedup_corpus.drop_duplicates(
    "family_signature", keep="first"
).copy()
analysis_corpus = family_dedup_corpus
consumer_voice_corpus = family_dedup_corpus.loc[
    family_dedup_corpus["register_bucket"].eq("consumer_narrative")
].copy()

linguistic_summary = pd.DataFrame({
    "all_rows": all_corpus[linguistic_flags].mean(),
    "exact_dedup": exact_dedup_corpus[linguistic_flags].mean(),
    "family_dedup": family_dedup_corpus[linguistic_flags].mean(),
    "consumer_voice_family_dedup": consumer_voice_corpus[linguistic_flags].mean(),
}).sort_values("family_dedup", ascending=False)

display(linguistic_summary.style.format("{:.1%}"))

density_summary = (
    family_dedup_corpus
      .groupby("register_bucket")[density_columns]
      .median()
      .T
)
density_summary.index.name = "median matches per 100 words"
display(density_summary.style.format("{:.2f}"))


### 6b. Parser-ready linguistic features

V4 introduces a parser hook for dependency-backed analysis. In this local environment, spaCy is optional: if `spacy` and an English pipeline such as `en_core_web_sm` are unavailable, the notebook records `parser_backend = heuristic_fallback` and emits conservative, auditable approximations. This matters methodologically: fallback features are useful for sampling and seed construction, but they should not be reported as parser-grade prevalence estimates.

The feature set targets FinDisputeEval failure modes: negation scope/focus, agentless passive financial events, self-repair/correction, temporal roles, speech-act layers, narrative-stage order, language variation, and semantic redaction types.


In [ ]:
try:
    import spacy  # type: ignore
    from spacy.util import is_package  # type: ignore
    if is_package("en_core_web_sm"):
        NLP = spacy.load("en_core_web_sm", disable=["ner"])
        PARSER_BACKEND = "spacy_en_core_web_sm"
    else:
        NLP = None
        PARSER_BACKEND = "heuristic_fallback_no_spacy_model"
except Exception as exc:
    NLP = None
    PARSER_BACKEND = f"heuristic_fallback_no_spacy:{type(exc).__name__}"

PARSER_STATUS = {
    "backend": PARSER_BACKEND,
    "dependency_parse_available": bool(NLP),
    "scope_note": (
        "Dependency-backed features are available only when spaCy and en_core_web_sm "
        "are installed. Current run uses conservative heuristics if unavailable."
    ),
}
print(json.dumps(PARSER_STATUS, indent=2))

SENTENCE_RE = re.compile(r"[^.!?\n]+(?:[.!?]+|$)")
NEGATION_CUE_RE = re.compile(
    r"\b(?:no|not|never|neither|nor|without|cannot|can't|couldn't|didn't|doesn't|"
    r"don't|wasn't|weren't|isn't|aren't|won't|wouldn't|haven't|hasn't)\b",
    re.I,
)
AUTHORIZATION_FOCUS_RE = re.compile(
    r"\b(?:authori[sz]e\w*|approve\w*|permission|consent|recognize\w*|made|make)\b",
    re.I,
)
FINANCIAL_FOCUS_RE = re.compile(
    r"\b(?:charge|transaction|purchase|payment|transfer|withdrawal|debit|credit|"
    r"account|card|loan|debt|collection|report|fee|interest|zelle|ach|wire)\b",
    re.I,
)
FINANCIAL_PASSIVE_RE = re.compile(
    r"\b(?:was|were|is|are|been|being|got)\s+(?:not\s+)?"
    r"(?P<verb>charged|debited|credited|removed|withdrawn|transferred|frozen|closed|"
    r"reported|declined|denied|reversed|posted|assessed|blocked|suspended|garnished|"
    r"misapplied|stolen|taken|opened|cancelled|canceled|rejected|compromised|hacked)\b",
    re.I,
)
REPAIR_PATTERNS = [
    re.compile(r"\bnot\s+(?P<old>[^,.!?;]{1,60}?)\s+but\s+(?P<new>[^,.!?;]{1,60})", re.I),
    re.compile(r"\b(?P<new>debit card|credit card|checking|savings|zelle|ach|wire|[A-Z][\w&.-]+),?\s+not\s+(?P<old>debit card|credit card|checking|savings|zelle|ach|wire|[A-Z][\w&.-]+)\b", re.I),
    re.compile(r"\b(?:actually|rather|I meant|I mean|correction)\b.{0,80}", re.I),
]
TIME_TOKEN_RE_V4 = re.compile(
    r"\[DATE\]|\b(?:yesterday|today|last (?:week|month|year|night)|"
    r"this (?:week|month)|\d+ days? ago|recently|since|before|after)\b",
    re.I,
)
CONTRASTIVE_NEGATION_RE = re.compile(
    r"\bnot\b[^.!?\n]{0,120}\bbut\b|\bno,?\s+[^.!?\n]{0,120}\bnot\b",
    re.I,
)
TEMPORAL_DISCOVERY_RE = re.compile(r"\b(?:noticed|discovered|found|saw|realized|became aware)\b", re.I)
TEMPORAL_TRANSACTION_RE = re.compile(r"\b(?:happened|charged|posted|withdrew|withdrawn|transferred|debited|purchase|transaction)\b", re.I)
TEMPORAL_REPORT_RE = re.compile(r"\b(?:reported|called|filed|notified|contacted|submitted|complained)\b", re.I)
TEMPORAL_STATEMENT_RE = re.compile(r"\b(?:statement|bill|invoice)\b", re.I)

SPEECH_ACT_PATTERNS = {
    "complaint": re.compile(r"\b(?:complaint|complain|unacceptable|wrong|error|mistake|problem)\b", re.I),
    "request_information": re.compile(r"\b(?:why|explain|would like to know|need someone to explain|please advise)\b", re.I),
    "request_action": re.compile(r"\b(?:please|I request|I ask|I want|I need|refund|reverse|remove|correct|investigate|resolve)\b", re.I),
    "demand_action": re.compile(r"\b(?:I demand|must|should|needs to be fixed|expect a full refund|immediately)\b", re.I),
    "dispute_initiation": re.compile(r"\b(?:dispute|disputing|unauthorized|not authorized|fraudulent|do not recognize)\b", re.I),
    "threat_escalation": re.compile(r"\b(?:attorney|lawyer|lawsuit|sue|legal action|regulator|CFPB|BBB|media)\b", re.I),
    "legal_performative": re.compile(r"\b(?:pursuant to|hereby|demand validation|cease and desist|FCRA|FDCPA|FCBA|EFTA|Regulation [EZ])\b", re.I),
    "emotional_evaluation": re.compile(r"\b(?:angry|upset|frustrated|ridiculous|stress|distress|unacceptable|furious)\b", re.I),
    "rhetorical_complaint": re.compile(r"\b(?:is this a joke|how is this (?:fair|legal|possible)|what kind of)\b", re.I),
    "indirect_request": re.compile(r"\b(?:I would like to know|I am hoping|I hope you can|can you|could you|would you)\b", re.I),
}
STAGE_PATTERNS_V4 = {
    "opening": re.compile(r"\b(?:I am filing|I am submitting|I am writing|this complaint|I need to dispute|I want to report)\b", re.I),
    "event": re.compile(r"\b(?:charged|withdrew|removed|transferred|payment|purchase|transaction|account was|card was|reported|collection)\b", re.I),
    "attribution": re.compile(r"\b(?:because|due to|merchant|company|bank|fraud|scam|stole|stolen|hacked|identity theft|error|mistake)\b", re.I),
    "impact": re.compile(r"\b(?:rent|food|medicine|bills?|financial hardship|damag\w* my credit|unable to|could not access|caused me|stress|distress)\b", re.I),
    "request": SPEECH_ACT_PATTERNS["request_action"],
    "coda": re.compile(r"(?:can you help|please help|thank you|sincerely|I look forward to|please advise)[.!?\s]*$", re.I),
}
SPANISH_ANCHORS = re.compile(r"\b(?:no reconozco|mi cuenta|mi tarjeta|por favor|mi dinero|no fui yo|fraude|ayuda|banco)\b", re.I)
AAVE_L1_TRANSFER_RE = re.compile(
    r"\b(?:ain't|aint|they be|it be|I be|I seen|I pay yesterday|I see charge|"
    r"charge on card|disputar|disputear)\b",
    re.I,
)
NON_ASCII_LATIN_RE = re.compile(r"[À-ÖØ-öø-ÿ]")
REDACTION_TYPE_PATTERNS = {
    "ACCOUNT_LAST4": re.compile(r"\b(?:account|acct)\s+(?:ending|number|#)?\s*\[REDACTED\]", re.I),
    "CARD_LAST4": re.compile(r"\b(?:card|debit|credit)\s+(?:ending|number|#)?\s*\[REDACTED\]", re.I),
    "PERSON_NAME": re.compile(r"\b(?:consumer|name|mr\.?|mrs\.?|ms\.?)\s*[:#]?\s*\[REDACTED\]", re.I),
    "ADDRESS": re.compile(r"\b(?:address|street|city|zip)\s*[:#]?\s*\[REDACTED\]", re.I),
    "EMAIL": re.compile(r"\b(?:email|e-mail)\s*[:#]?\s*\[REDACTED\]", re.I),
    "PHONE": re.compile(r"\b(?:phone|telephone|mobile|cell)\s*[:#]?\s*\[REDACTED\]", re.I),
    "SSN_OR_TAX_ID": re.compile(r"\b(?:ssn|social security|tax id|ein)\s*[:#]?\s*\[REDACTED\]", re.I),
}


def sentence_for_offset(text: str, offset: int) -> tuple[int, str, int, int]:
    text = str(text)
    left = max(text.rfind(".", 0, offset), text.rfind("!", 0, offset), text.rfind("?", 0, offset), text.rfind("\n", 0, offset))
    start = 0 if left < 0 else left + 1
    right_candidates = [pos for pos in [text.find(".", offset), text.find("!", offset), text.find("?", offset), text.find("\n", offset)] if pos >= 0]
    end = min(right_candidates) + 1 if right_candidates else min(len(text), offset + 220)
    sent_idx = len(re.findall(r"[.!?\n]", text[:start]))
    return sent_idx, text[start:end].strip(), start, end


def infer_repair_slot(old_value: str, new_value: str) -> str:
    combined = f"{old_value} {new_value}".lower()
    if re.search(r"\b(?:debit|credit|prepaid|card)\b", combined):
        return "card_type"
    if re.search(r"\b(?:checking|savings|account)\b", combined):
        return "account_type"
    if re.search(r"\b(?:\[AMOUNT\]|\$|amount|dollar)\b", combined):
        return "amount"
    if re.search(r"\b(?:\[DATE\]|yesterday|today|week|month|year)\b", combined):
        return "date"
    if re.search(r"\b(?:merchant|amazon|walmart|uber|store|company)\b", combined):
        return "merchant"
    if re.search(r"\b(?:authorized|unauthorized|fraud|duplicate|subscription)\b", combined):
        return "claim_type"
    return "unspecified"


def extract_linguistic_frame(text: str) -> dict:
    text = str(text)
    neg = NEGATION_CUE_RE.search(text)
    neg_scope = ""
    neg_predicate = ""
    neg_focus = ""
    if neg:
        _, sent, sent_start, _ = sentence_for_offset(text, neg.start())
        scope_start = max(0, neg.start() - sent_start)
        neg_scope = sent[scope_start:][:180].strip()
        after = sent[scope_start:]
        pred = AUTHORIZATION_FOCUS_RE.search(after) or FINANCIAL_FOCUS_RE.search(after)
        neg_predicate = pred.group(0).lower() if pred else "unknown"
        focus = FINANCIAL_FOCUS_RE.search(after)
        neg_focus = focus.group(0).lower() if focus else "unknown"
    passive = FINANCIAL_PASSIVE_RE.search(text)
    passive_window = text[passive.start(): passive.end() + 80] if passive else ""
    agent_present = bool(passive and re.search(r"\bby\b", passive_window, re.I))
    repair_type = "none_detected"
    repair_old = ""
    repair_new = ""
    repair_slot = ""
    for pattern in REPAIR_PATTERNS:
        match = pattern.search(text)
        if match:
            repair_type = "contrastive_replacement" if "old" in match.groupdict() else "explicit_repair_marker"
            repair_old = (match.groupdict().get("old") or "").strip()
            repair_new = (match.groupdict().get("new") or "").strip()
            repair_slot = infer_repair_slot(repair_old, repair_new)
            break
    date_mentions = []
    for match in TIME_TOKEN_RE_V4.finditer(text):
        window = text[max(0, match.start() - 80): match.end() + 80]
        if TEMPORAL_DISCOVERY_RE.search(window):
            role, confidence = "discovery_date", "medium"
        elif TEMPORAL_TRANSACTION_RE.search(window):
            role, confidence = "transaction_date", "medium"
        elif TEMPORAL_REPORT_RE.search(window):
            role, confidence = "report_date", "medium"
        elif TEMPORAL_STATEMENT_RE.search(window):
            role, confidence = "statement_date", "medium"
        else:
            role, confidence = "unknown_date", "low"
        date_mentions.append({"text": match.group(0), "role": role, "confidence": confidence, "start": match.start()})
    speech_layers = [name for name, pattern in SPEECH_ACT_PATTERNS.items() if pattern.search(text)]
    stage_hits = []
    for stage, pattern in STAGE_PATTERNS_V4.items():
        match = pattern.search(text)
        if match:
            sent_idx, _, _, _ = sentence_for_offset(text, match.start())
            stage_hits.append({"stage": stage, "char_offset": match.start(), "sentence_index": sent_idx})
    stage_hits = sorted(stage_hits, key=lambda item: item["char_offset"])
    stage_order = "|".join(item["stage"] for item in stage_hits) if stage_hits else "none_detected"
    stage_offsets = {item["stage"]: item["char_offset"] for item in stage_hits}
    event_pos = stage_offsets.get("event")
    impact_pos = stage_offsets.get("impact")
    if event_pos is not None and impact_pos is not None:
        sequence_type = "event_before_impact" if event_pos < impact_pos else "impact_before_event"
    elif event_pos is not None:
        sequence_type = "event_without_impact"
    elif impact_pos is not None:
        sequence_type = "impact_without_event"
    else:
        sequence_type = "no_event_or_impact"
    redaction_types = [name for name, pattern in REDACTION_TYPE_PATTERNS.items() if pattern.search(text)]
    return {
        "negation_cue": neg.group(0).lower() if neg else "",
        "negated_predicate": neg_predicate,
        "negation_scope_text": neg_scope,
        "negation_focus_entity": neg_focus,
        "embedded_negation_risk": bool(re.search(r"\b(?:did not|didn't|not)\s+(?:say|claim|state|mean)\b", text, re.I)),
        "contrastive_negation": bool(CONTRASTIVE_NEGATION_RE.search(text)),
        "polarity_flip_risk": bool(neg and (AUTHORIZATION_FOCUS_RE.search(text) or re.search(r"\bbut\b", text, re.I))),
        "passive_event_verb": passive.group("verb").lower() if passive else "",
        "passive_agent_present": agent_present,
        "agentless_financial_passive": bool(passive and not agent_present),
        "repair_type": repair_type,
        "repair_target_slot": repair_slot,
        "old_value_candidate": repair_old,
        "new_value_candidate": repair_new,
        "requires_route_recompute": repair_slot in {"card_type", "account_type", "claim_type"},
        "date_mentions_json": json.dumps(date_mentions, ensure_ascii=False),
        "date_role_count": len({item["role"] for item in date_mentions if item["role"] != "unknown_date"}),
        "date_conflict_flag": len({item["role"] for item in date_mentions}) >= 2,
        "temporal_role_summary": "|".join(sorted({item["role"] for item in date_mentions})) if date_mentions else "none_detected",
        "speech_act_layers": "|".join(speech_layers) if speech_layers else "none_detected",
        "stage_order_pattern": stage_order,
        "stage_offsets_json": json.dumps(stage_hits, ensure_ascii=False),
        "narrative_sequence_type": sequence_type,
        "nonlinear_narrative_flag": sequence_type == "impact_before_event" or bool(re.search(r"\b(?:but|however|although|before|after|earlier|later)\b", text, re.I)),
        "spanish_anchor_count": len(SPANISH_ANCHORS.findall(text)),
        "non_ascii_latin_count": len(NON_ASCII_LATIN_RE.findall(text)),
        "language_variation_candidate": bool(SPANISH_ANCHORS.search(text) or AAVE_L1_TRANSFER_RE.search(text) or NON_ASCII_LATIN_RE.search(text)),
        "redaction_type_candidates": "|".join(redaction_types) if redaction_types else "none_inferred",
    }


frames = df["narrative_normalized"].map(extract_linguistic_frame).apply(pd.Series)
for column in frames.columns:
    df[column] = frames[column]
df["parser_backend"] = PARSER_BACKEND


def count_pipe_values(series: pd.Series) -> Counter:
    counter = Counter()
    for value in series.dropna().astype(str):
        for item in value.split("|"):
            if item and item != "none_detected":
                counter[item] += 1
    return counter


language_coverage_note = {
    "spanish_anchor_phrases": ["no reconozco", "mi cuenta", "mi tarjeta", "por favor", "mi dinero", "no fui yo", "fraude", "ayuda", "banco"],
    "coverage_limit": "Lower-bound detector only. Does not fully detect code-switching, AAVE, L1 transfer, or all non-English text.",
}

all_corpus = df.loc[df["narrative_normalized"].str.len().gt(0)].copy()
exact_dedup_corpus = all_corpus.drop_duplicates("narrative_hash", keep="first").copy()
family_dedup_corpus = exact_dedup_corpus.drop_duplicates("family_signature", keep="first").copy()
analysis_corpus = family_dedup_corpus
consumer_voice_corpus = family_dedup_corpus.loc[family_dedup_corpus["register_bucket"].eq("consumer_narrative")].copy()

parser_feature_summary = pd.DataFrame({
    "rate": analysis_corpus[[
        "polarity_flip_risk", "agentless_financial_passive", "requires_route_recompute",
        "date_conflict_flag", "nonlinear_narrative_flag", "language_variation_candidate"
    ]].mean(),
    "count": analysis_corpus[[
        "polarity_flip_risk", "agentless_financial_passive", "requires_route_recompute",
        "date_conflict_flag", "nonlinear_narrative_flag", "language_variation_candidate"
    ]].sum(),
}).sort_values("rate", ascending=False)
display(parser_feature_summary.style.format({"rate": "{:.1%}"}))
print("Top speech-act layers:", count_pipe_values(analysis_corpus["speech_act_layers"]).most_common(12))
print("Redaction type candidates:", count_pipe_values(analysis_corpus["redaction_type_candidates"]).most_common(12))


## 6c. FinDisputeEval scenario seed candidates

V4 turns corpus observations into structured scenario seed candidates. These records are not legal conclusions; they are intake hypotheses for synthetic dialogue generation, stress testing, and human review. The important shift is from `lx_* = True` to route-relevant fields: claim type, payment rail, card/account type, missing slots, risk flags, and route hint.


In [ ]:
CLAIM_PATTERNS = [
    ("identity_theft_fraud", re.compile(r"\b(?:identity theft|stolen identity|fraud|fraudulent|scam|account takeover|hacked)\b", re.I)),
    ("unauthorized", re.compile(r"\b(?:unauthorized|not authori[sz]ed|did not authori[sz]e|without (?:my )?(?:permission|consent)|do not recognize|didn't recognize|no permission)\b", re.I)),
    ("duplicate", re.compile(r"\b(?:duplicate|double charged|charged twice|two charges|twice for the same|same charge twice)\b", re.I)),
    ("non_delivery", re.compile(r"\b(?:never received|not received|did not receive|goods? not delivered|service not provided|non[- ]delivery)\b", re.I)),
    ("subscription_recurring", re.compile(r"\b(?:subscription|recurring|monthly charge|membership|free trial|cancelled|canceled|auto[- ]?renew)\b", re.I)),
    ("merchant_dispute", re.compile(r"\b(?:merchant|retailer|store|vendor|refund|returned|return policy|defective|service issue)\b", re.I)),
    ("billing_error", re.compile(r"\b(?:billing error|wrong amount|incorrect charge|overcharged|charged in error|misapplied payment)\b", re.I)),
    ("fee_interest_dispute", re.compile(r"\b(?:fee|interest|overdraft|late fee|finance charge|penalty)\b", re.I)),
    ("debt_validation", re.compile(r"\b(?:debt validation|validate (?:this )?debt|debt collector|collection agency|FDCPA|cease and desist)\b", re.I)),
    ("credit_reporting_dispute", re.compile(r"\b(?:credit report|credit bureau|tradeline|score|Equifax|Experian|TransUnion|FCRA)\b", re.I)),
]
PAYMENT_RAIL_PATTERNS = [
    ("zelle", re.compile(r"\bZelle\b", re.I)),
    ("ach", re.compile(r"\bACH\b|automated clearing house", re.I)),
    ("wire", re.compile(r"\bwire transfer\b|\bwire\b", re.I)),
    ("debit_card", re.compile(r"\bdebit card\b", re.I)),
    ("credit_card", re.compile(r"\bcredit card\b", re.I)),
    ("prepaid_card", re.compile(r"\bprepaid card\b", re.I)),
    ("checking_account", re.compile(r"\bchecking account\b", re.I)),
    ("savings_account", re.compile(r"\bsavings account\b", re.I)),
    ("card_not_present", re.compile(r"\b(?:online|internet|card not present|CNP|over the phone)\b", re.I)),
]
SLOT_PATTERNS = {
    "amount": re.compile(r"\[AMOUNT\]|\$\s*\d|\b(?:dollars?|usd)\b", re.I),
    "transaction_date": re.compile(r"\[DATE\]|\b(?:yesterday|today|last (?:week|month|year|night)|\d+ days? ago)\b", re.I),
    "merchant": re.compile(r"\b(?:merchant|retailer|store|restaurant|vendor|Amazon|Walmart|Uber|Costco|Target|Apple|Google|PayPal)\b", re.I),
    "card_type": re.compile(r"\b(?:debit card|credit card|prepaid card)\b", re.I),
    "card_possession": re.compile(r"\b(?:card was stolen|lost my card|still have my card|card in my possession|never lost)\b", re.I),
    "account_type": re.compile(r"\b(?:checking account|savings account|deposit account)\b", re.I),
    "third_party_auth": re.compile(r"\b(?:family member|spouse|child|roommate|friend|authorized user|gave permission)\b", re.I),
    "merchant_contact": re.compile(r"\b(?:called|contacted|emailed|wrote to)\s+(?:the )?(?:merchant|store|company|vendor|retailer)\b", re.I),
    "police_or_identity_report": re.compile(r"\b(?:police report|FTC report|identity theft report|filed a report)\b", re.I),
    "customer_reported_to_bank_date": re.compile(r"\b(?:reported|called|notified|contacted)\s+(?:my )?(?:bank|credit union|card issuer|lender)\b", re.I),
}
RISK_FLAG_PATTERNS = {
    "identity_theft": CLAIM_PATTERNS[0][1],
    "legal_threat": re.compile(r"\b(?:attorney|lawyer|lawsuit|sue|legal action|pursuant to|FCRA|FDCPA|FCBA|EFTA)\b", re.I),
    "regulator_mention": re.compile(r"\b(?:CFPB|regulator|BBB|Better Business Bureau|attorney general)\b", re.I),
    "vulnerable_customer": re.compile(r"\b(?:elderly|disabled|fixed income|veteran|medical|hospital|unemployed)\b", re.I),
    "urgent_funds_hardship": re.compile(r"\b(?:rent|food|medicine|utilities|eviction|paycheck|unable to pay|financial hardship)\b", re.I),
    "repeated_failed_contact": re.compile(r"\b(?:called|contacted|emailed).{0,60}\b(?:multiple|several|many|repeated|again)\b|\b(?:no response|never responded|kept calling)\b", re.I | re.S),
    "bank_refused_escalation": re.compile(r"\b(?:refused|denied|would not help|closed my claim|no investigation|ignored)\b", re.I),
    "fraud_scam": re.compile(r"\b(?:fraud|fraudulent|scam|phishing|spoof|imposter)\b", re.I),
    "account_takeover": re.compile(r"\b(?:account takeover|hacked|password changed|locked out|unauthorized login)\b", re.I),
    "card_stolen": re.compile(r"\b(?:card was stolen|stolen card|lost card)\b", re.I),
    "emotional_distress": LINGUISTIC_PATTERNS["lx_emotion"],
}


def first_matching_label(text: str, patterns: list[tuple[str, re.Pattern]], default: str = "unknown_ambiguous") -> str:
    for label, pattern in patterns:
        if pattern.search(text):
            return label
    return default


def all_matching_labels(text: str, patterns: dict[str, re.Pattern]) -> list[str]:
    return [label for label, pattern in patterns.items() if pattern.search(text)]


def route_hint(claim_type: str, payment_rail: str) -> str:
    if claim_type == "credit_reporting_dispute":
        return "FCRA_credit_reporting_review"
    if claim_type == "debt_validation":
        return "FDCPA_debt_validation_review"
    if payment_rail == "credit_card":
        return "Reg_Z_FCBA_credit_card_billing_dispute"
    if payment_rail in {"debit_card", "prepaid_card", "checking_account", "savings_account", "ach", "zelle", "wire"}:
        return "Reg_E_or_transfer_review_requires_facts"
    if claim_type in {"unauthorized", "identity_theft_fraud", "duplicate", "billing_error"}:
        return "unknown_requires_payment_rail_and_card_type"
    return "nonfinal_triage_requires_human_review"


def build_scenario_seed(row: pd.Series) -> dict:
    text = str(row["narrative_normalized"])
    claim_type = first_matching_label(text, CLAIM_PATTERNS)
    payment_rail = first_matching_label(text, PAYMENT_RAIL_PATTERNS, default="unknown")
    card_type = payment_rail if payment_rail in {"debit_card", "credit_card", "prepaid_card"} else "unknown"
    present_slots = all_matching_labels(text, SLOT_PATTERNS)
    missing_slots = [slot for slot in SLOT_PATTERNS if slot not in present_slots]
    risk_flags = all_matching_labels(text, RISK_FLAG_PATTERNS)
    if bool(row.get("polarity_flip_risk", False)):
        risk_flags.append("authorization_or_negation_scope_risk")
    if bool(row.get("date_conflict_flag", False)):
        risk_flags.append("multi_role_temporal_risk")
    if float(row.get("redaction_density", 0.0)) > 0.15:
        risk_flags.append("high_redaction_sensitive_info")
    if payment_rail == "unknown" and claim_type in {"unauthorized", "duplicate", "billing_error", "identity_theft_fraud"}:
        risk_flags.append("route_unknown_payment_rail")
    risk_flags = sorted(set(risk_flags))
    route = route_hint(claim_type, payment_rail)
    review_reasons = []
    if claim_type == "unknown_ambiguous":
        review_reasons.append("claim type ambiguous")
    if "route_unknown_payment_rail" in risk_flags:
        review_reasons.append("payment rail/card type missing for routing")
    if bool(row.get("polarity_flip_risk", False)):
        review_reasons.append("negation scope may affect authorization claim")
    if len(missing_slots) >= 5:
        review_reasons.append("many intake slots missing")
    if bool(row.get("requires_route_recompute", False)):
        review_reasons.append("self-repair may change route")
    return {
        "complaint_id": str(row["Complaint ID"]),
        "source": "CFPB",
        "register_bucket": str(row["register_bucket"]),
        "claim_type_candidate": claim_type,
        "card_type_candidate": card_type,
        "payment_rail_candidate": payment_rail,
        "route_hint": route,
        "missing_slots": missing_slots,
        "present_slots": present_slots,
        "risk_flags": risk_flags,
        "linguistic_features": {
            "negation_scope": row.get("negation_scope_text", ""),
            "negated_predicate": row.get("negated_predicate", ""),
            "negation_focus_entity": row.get("negation_focus_entity", ""),
            "hedging": bool(row.get("lx_hedging", False)),
            "temporal_role_summary": row.get("temporal_role_summary", ""),
            "speech_act_layers": row.get("speech_act_layers", ""),
            "stage_order_pattern": row.get("stage_order_pattern", ""),
            "authorization_ambiguity": bool(row.get("polarity_flip_risk", False)),
        },
        "human_review_reason": "; ".join(review_reasons) if review_reasons else "routine_review",
    }


seed_records = df.apply(build_scenario_seed, axis=1)
seed_frame = pd.json_normalize(seed_records)
seed_frame.index = df.index
for column in ["claim_type_candidate", "card_type_candidate", "payment_rail_candidate", "route_hint", "human_review_reason"]:
    df[column] = seed_frame[column]
df["missing_slots"] = seed_records.map(lambda item: "|".join(item["missing_slots"]) if item["missing_slots"] else "none")
df["present_slots"] = seed_records.map(lambda item: "|".join(item["present_slots"]) if item["present_slots"] else "none")
df["risk_flags"] = seed_records.map(lambda item: "|".join(item["risk_flags"]) if item["risk_flags"] else "none")
df["scenario_seed_candidate"] = seed_records.map(lambda item: json.dumps(item, ensure_ascii=False))
df["missing_slot_count"] = df["missing_slots"].map(lambda value: 0 if value == "none" else len(str(value).split("|")))
df["risk_flag_count"] = df["risk_flags"].map(lambda value: 0 if value == "none" else len(str(value).split("|")))



def add_complexity_features(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame["negation_count"] = frame["narrative_normalized"].str.count(NEGATION_CUE_RE)
    frame["hedging_count"] = frame["narrative_normalized"].str.count(LINGUISTIC_PATTERNS["lx_hedging"])
    frame["temporal_anchor_count"] = frame["date_mentions_json"].map(lambda value: len(json.loads(value)) if isinstance(value, str) and value else 0)
    frame["repair_candidate_count"] = frame["repair_type"].ne("none_detected").astype(int)
    frame["implicit_slot_penalty"] = frame["missing_slot_count"].clip(upper=8)
    register_weight = frame["register_bucket"].map({
        "consumer_narrative": 1.0,
        "legal_formal": 1.5,
        "pasted_correspondence": 1.2,
        "template_form": 1.2,
        "template_letter_family": 1.5,
    }).fillna(1.0)
    frame["complexity_score"] = (
        frame["negation_count"]
        + frame["hedging_count"]
        + frame["temporal_anchor_count"]
        + frame["repair_candidate_count"] * 2
        + frame["polarity_flip_risk"].astype(int) * 2
        + frame["lx_escalation_signal"].astype(int)
        + frame["implicit_slot_penalty"] * 0.5
        + frame["nonlinear_narrative_flag"].astype(int) * 2
        + register_weight
    )
    frame["complexity_tier"] = pd.cut(
        frame["complexity_score"],
        bins=[-np.inf, 3, 7, 12, np.inf],
        labels=["simple", "medium", "complex", "adversarial_high_risk"],
    )
    return frame

df = add_complexity_features(df)

all_corpus = df.loc[df["narrative_normalized"].str.len().gt(0)].copy()
exact_dedup_corpus = all_corpus.drop_duplicates("narrative_hash", keep="first").copy()
family_dedup_corpus = exact_dedup_corpus.drop_duplicates("family_signature", keep="first").copy()
analysis_corpus = family_dedup_corpus
consumer_voice_corpus = family_dedup_corpus.loc[family_dedup_corpus["register_bucket"].eq("consumer_narrative")].copy()

scenario_seed_path = OUTPUT_DIR / "cfpb_scenario_seed_candidates_v4.jsonl"
with scenario_seed_path.open("w", encoding="utf-8") as handle:
    for item in analysis_corpus["scenario_seed_candidate"]:
        handle.write(item + "\n")

scenario_summary = pd.DataFrame({
    "claim_type_rate": analysis_corpus["claim_type_candidate"].value_counts(normalize=True),
    "claim_type_count": analysis_corpus["claim_type_candidate"].value_counts(),
}).fillna(0)
display(scenario_summary)
display(analysis_corpus["payment_rail_candidate"].value_counts().to_frame("count"))
display(analysis_corpus["route_hint"].value_counts().to_frame("count"))
print(f"Saved scenario seeds: {scenario_seed_path}")


In [ ]:
phenomenon_by_product = (
    family_dedup_corpus
      .groupby("Product")[linguistic_flags]
      .mean()
      .assign(n=family_dedup_corpus.groupby("Product").size())
      .sort_values("n", ascending=False)
)

display(phenomenon_by_product.head(15)
        .style.format({column: "{:.1%}" for column in linguistic_flags}))


## 7. Concordance / KWIC review

Use KWIC to inspect how a word or phrase functions in context before converting a heuristic into an annotation rule.

Regex patterns are supported, all matches per document are retained, and rows are sampled with a fixed seed rather than taken in file order. File order correlates with date and company and can silently bias the inspection sample.


In [ ]:
def kwic(data: pd.DataFrame, term: str | re.Pattern, width: int = 90,
         limit: int = 20, seed: int = 20260609,
         text_column: str = "narrative_normalized") -> pd.DataFrame:
    """Keyword-in-context with regex support and seeded random sampling."""
    pattern = term if isinstance(term, re.Pattern) else re.compile(re.escape(term), re.I)
    matched = data.loc[data[text_column].str.contains(pattern, na=False)]
    if matched.empty:
        return pd.DataFrame(columns=["Complaint ID", "Product", "left", "keyword", "right"])
    sampled = matched.sample(n=min(limit, len(matched)), random_state=seed)

    rows = []
    for _, row in sampled.iterrows():
        text = str(row[text_column])
        for match in pattern.finditer(text):
            rows.append({
                "Complaint ID": row["Complaint ID"],
                "Product": row["Product"],
                "left": text[max(0, match.start() - width):match.start()],
                "keyword": match.group(0),
                "right": text[match.end():match.end() + width],
            })
            if len(rows) >= limit:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)


kwic(analysis_corpus, "authorize", limit=15)


### 7b. Pattern audit with positive and negative controls

Before any pattern's hit-rate is quoted as a prevalence estimate, estimate its **precision**: sample matched spans, eyeball them, and record true/false positives. This cell exports one audit file with up to `AUDIT_PER_PATTERN` random matches per pattern, each with the matched span and local context, plus an empty `is_true_positive` column for manual marking.

A pattern with measured precision becomes a citable instrument ("`lx_hedging`, precision ≈ 0.84 on a 25-hit audit, n=..."); a pattern without one is a guess.


In [ ]:
AUDIT_POSITIVES_PER_PATTERN = 20
AUDIT_NEGATIVES_PER_PATTERN = 10
AUDIT_SEED = 20260609

audit_rows = []
for flag, pattern in LINGUISTIC_PATTERNS.items():
    match_mask = analysis_corpus["narrative_normalized"].str.contains(pattern, na=False)
    matched = analysis_corpus.loc[match_mask]
    unmatched = analysis_corpus.loc[~match_mask]
    positive_sample = matched.sample(
        n=min(AUDIT_POSITIVES_PER_PATTERN, len(matched)), random_state=AUDIT_SEED
    ) if not matched.empty else matched
    negative_sample = unmatched.sample(
        n=min(AUDIT_NEGATIVES_PER_PATTERN, len(unmatched)), random_state=AUDIT_SEED + 1
    ) if not unmatched.empty else unmatched
    for _, row in positive_sample.iterrows():
        text = str(row["narrative_normalized"])
        match = pattern.search(text)
        if match is None:
            continue
        audit_rows.append({
            "pattern": flag,
            "Complaint ID": row["Complaint ID"],
            "register_bucket": row["register_bucket"],
            "matched_span": match.group(0),
            "context": text[max(0, match.start() - 80):match.end() + 80],
            "audit_type": "detected_positive",
            "manual_correct": "",   # fill in manually: 1 / 0
            "notes": "",
        })
    for _, row in negative_sample.iterrows():
        text = str(row["narrative_normalized"])
        audit_rows.append({
            "pattern": flag,
            "Complaint ID": row["Complaint ID"],
            "register_bucket": row["register_bucket"],
            "matched_span": "",
            "context": text[:240],
            "audit_type": "negative_control",
            "manual_correct": "",   # 1 if phenomenon is truly absent, else 0
            "notes": "",
        })

pattern_audit = pd.DataFrame(audit_rows)
pattern_audit_path = OUTPUT_DIR / "cfpb_pattern_audit_v4.csv"
pattern_audit.to_csv(pattern_audit_path, index=False, encoding="utf-8-sig")

print(f"Audit rows: {len(pattern_audit):,} across {pattern_audit['pattern'].nunique()} patterns")
display(pd.crosstab(pattern_audit["pattern"], pattern_audit["audit_type"]))
print(f"Saved: {pattern_audit_path}")
pattern_audit.head(8)


### 7c. Register audit sample

Register assignment is still rule-based. V4 exports a small stratified audit file so a human reviewer can mark `manual_register` and `manual_correct`, then compute a confusion matrix once labels exist. This keeps register evidence useful without treating it as validated truth.


In [ ]:
REGISTER_AUDIT_N_PER_BUCKET = 20
REGISTER_AUDIT_SEED = 20260629
register_audit_parts = []
for offset, register in enumerate(sorted(analysis_corpus["register_bucket"].unique())):
    pool = analysis_corpus.loc[analysis_corpus["register_bucket"].eq(register)]
    register_audit_parts.append(
        pool.sample(n=min(REGISTER_AUDIT_N_PER_BUCKET, len(pool)), random_state=REGISTER_AUDIT_SEED + offset)
    )

register_audit = pd.concat(register_audit_parts, ignore_index=True)
register_audit = register_audit[[
    "Complaint ID", "Product", "Issue", "register_bucket", "register_confidence",
    "register_evidence", "word_count", "narrative_normalized"
]].copy()
register_audit.insert(0, "register_audit_id", [f"REG-{i:03d}" for i in range(1, len(register_audit) + 1)])
register_audit["manual_register"] = ""
register_audit["manual_correct"] = ""
register_audit["notes"] = ""
register_audit_path = OUTPUT_DIR / "cfpb_register_audit_sample_v4.csv"
register_audit.to_csv(register_audit_path, index=False, encoding="utf-8-sig")
print(f"Register audit rows: {len(register_audit):,}")
display(register_audit["register_bucket"].value_counts().to_frame("sample_count"))
print(f"Saved: {register_audit_path}")


## 8. TF-IDF analysis

TF-IDF is calculated on the deduplicated corpus with unigram and bigram features. The stopword list deliberately retains negation and modal terms such as `not`, `never`, `might`, and `should`, because removing them would damage the linguistic analysis. CFPB placeholders are excluded from the TF-IDF input.


In [ ]:
semantic_function_words = {
    "no", "not", "never", "nor", "without", "cannot",
    "may", "might", "must", "should", "could", "would",
}
tfidf_stop_words = sorted(set(ENGLISH_STOP_WORDS) - semantic_function_words)

analysis_corpus["tfidf_text"] = (
    analysis_corpus["narrative_noplaceholder"]
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=tfidf_stop_words,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.85,
    max_features=10_000,
    sublinear_tf=True,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z'-]{1,}\b",
)

tfidf_matrix = tfidf_vectorizer.fit_transform(analysis_corpus["tfidf_text"])
tfidf_terms = np.asarray(tfidf_vectorizer.get_feature_names_out())
mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).ravel()
document_frequency = np.asarray((tfidf_matrix > 0).sum(axis=0)).ravel()

top_tfidf = (
    pd.DataFrame({
        "term": tfidf_terms,
        "mean_tfidf": mean_tfidf,
        "document_frequency": document_frequency,
    })
      .sort_values("mean_tfidf", ascending=False)
      .head(30)
      .reset_index(drop=True)
)

print(f"TF-IDF matrix: {tfidf_matrix.shape[0]:,} documents x {tfidf_matrix.shape[1]:,} features")
display(top_tfidf)

top_plot = top_tfidf.head(20).sort_values("mean_tfidf")
ax = top_plot.plot.barh(
    x="term", y="mean_tfidf", figsize=(10, 7), legend=False,
    title="Top TF-IDF terms in the deduplicated corpus",
)
ax.set_xlabel("Mean TF-IDF")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
def top_terms_for_mask(mask: np.ndarray, n: int = 15) -> pd.DataFrame:
    group_scores = np.asarray(tfidf_matrix[mask].mean(axis=0)).ravel()
    indices = np.argsort(group_scores)[::-1][:n]
    return pd.DataFrame({
        "term": tfidf_terms[indices],
        "mean_tfidf": group_scores[indices],
    })


register_top_terms = []
register_values = analysis_corpus["register_bucket"].to_numpy()

for register in sorted(analysis_corpus["register_bucket"].unique()):
    register_terms = top_terms_for_mask(register_values == register, n=15)
    register_terms.insert(0, "register_bucket", register)
    register_terms.insert(1, "rank", range(1, len(register_terms) + 1))
    register_top_terms.append(register_terms)

register_top_terms = pd.concat(register_top_terms, ignore_index=True)
display(
    register_top_terms.pivot(index="rank", columns="register_bucket", values="term")
)

global_tfidf_output_path = OUTPUT_DIR / "cfpb_tfidf_global_top_terms_v4.csv"
register_tfidf_output_path = OUTPUT_DIR / "cfpb_tfidf_register_top_terms_v4.csv"
top_tfidf.to_csv(global_tfidf_output_path, index=False, encoding="utf-8-sig")
register_top_terms.to_csv(register_tfidf_output_path, index=False, encoding="utf-8-sig")
print(f"Saved: {global_tfidf_output_path}")
print(f"Saved: {register_tfidf_output_path}")


### 8b. Register keyness via log-odds with informative Dirichlet prior

Comparing groups by *mean TF-IDF* mostly resurfaces globally frequent terms. The corpus-linguistics standard for "what is distinctive of register A vs. register B" is a **keyness** statistic; here we use the log-odds ratio with an informative Dirichlet prior (Monroe, Colaresi & Quinn 2008), which shrinks rare-word noise using the whole corpus as the prior and yields a z-score per term.

Reading: large positive z → characteristic of consumer-voice narratives; large negative z → characteristic of legal-formal register. Expect deictic/affective/dysfluent material on the consumer side and citation/performative-legal material on the formal side — a quantitative footing for the register split used in sampling.


In [ ]:
count_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words=tfidf_stop_words,
    ngram_range=(1, 1),
    min_df=5,
    max_features=20_000,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z'-]{1,}\b",
)
count_matrix = count_vectorizer.fit_transform(analysis_corpus["tfidf_text"])
count_terms = np.asarray(count_vectorizer.get_feature_names_out())

mask_a = analysis_corpus["register_bucket"].eq("consumer_narrative").to_numpy()
mask_b = analysis_corpus["register_bucket"].eq("legal_formal").to_numpy()

y_a = np.asarray(count_matrix[mask_a].sum(axis=0)).ravel().astype(float)
y_b = np.asarray(count_matrix[mask_b].sum(axis=0)).ravel().astype(float)
alpha = np.asarray(count_matrix.sum(axis=0)).ravel().astype(float)  # corpus-wide prior
alpha = alpha * (1000.0 / alpha.sum())                              # prior strength a0 = 1000

n_a, n_b, a0 = y_a.sum(), y_b.sum(), alpha.sum()
log_odds_a = np.log((y_a + alpha) / (n_a + a0 - y_a - alpha))
log_odds_b = np.log((y_b + alpha) / (n_b + a0 - y_b - alpha))
delta = log_odds_a - log_odds_b
variance = 1.0 / (y_a + alpha) + 1.0 / (y_b + alpha)
z_scores = delta / np.sqrt(variance)

keyness = pd.DataFrame({
    "term": count_terms,
    "z": z_scores,
    "count_consumer": y_a.astype(int),
    "count_legal": y_b.astype(int),
    "n_docs_consumer": int(mask_a.sum()),
    "n_docs_legal": int(mask_b.sum()),
    "n_tokens_consumer": int(n_a),
    "n_tokens_legal": int(n_b),
})

print(f"Consumer documents: {mask_a.sum():,}; legal-formal documents: {mask_b.sum():,}")
print("Most characteristic of CONSUMER-VOICE narratives:")
display(keyness.sort_values("z", ascending=False).head(20).reset_index(drop=True))
print("Most characteristic of LEGAL-FORMAL register:")
display(keyness.sort_values("z").head(20).reset_index(drop=True))

keyness_path = OUTPUT_DIR / "cfpb_register_keyness_logodds_v4.csv"
keyness.sort_values("z", ascending=False).to_csv(keyness_path, index=False, encoding="utf-8-sig")
print(f"Saved: {keyness_path}")


### 8c. Collocation diagnostics

Single-term TF-IDF is useful for orientation but weak for FinDisputeEval design. V4 adds frequent bigram/trigram collocations and a consumer-vs-legal keyness scatter so formulaic legal language and consumer dispute constructions are easier to inspect.


In [ ]:
collocation_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words=tfidf_stop_words,
    ngram_range=(2, 3),
    min_df=5,
    max_features=30_000,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z'-]{1,}\b",
)
collocation_matrix = collocation_vectorizer.fit_transform(analysis_corpus["tfidf_text"])
collocation_terms = np.asarray(collocation_vectorizer.get_feature_names_out())
collocation_counts = np.asarray(collocation_matrix.sum(axis=0)).ravel().astype(int)
collocation_docfreq = np.asarray((collocation_matrix > 0).sum(axis=0)).ravel().astype(int)
collocations = pd.DataFrame({
    "collocation": collocation_terms,
    "term_count": collocation_counts,
    "document_frequency": collocation_docfreq,
}).sort_values(["document_frequency", "term_count"], ascending=False)
collocation_path = OUTPUT_DIR / "cfpb_collocations_v4.csv"
collocations.head(300).to_csv(collocation_path, index=False, encoding="utf-8-sig")
display(collocations.head(30))
print(f"Saved: {collocation_path}")

keyness_scatter = keyness.copy()
keyness_scatter["freq_consumer_per_10k"] = keyness_scatter["count_consumer"] / keyness_scatter["n_tokens_consumer"].clip(lower=1) * 10_000
keyness_scatter["freq_legal_per_10k"] = keyness_scatter["count_legal"] / keyness_scatter["n_tokens_legal"].clip(lower=1) * 10_000
keyness_scatter_path = OUTPUT_DIR / "cfpb_keyness_scatter_data_v4.csv"
keyness_scatter.to_csv(keyness_scatter_path, index=False, encoding="utf-8-sig")

plot_data = keyness_scatter.loc[(keyness_scatter["count_consumer"] + keyness_scatter["count_legal"]) >= 20].copy()
plt.figure(figsize=(8, 6))
plt.scatter(
    plot_data["freq_consumer_per_10k"],
    plot_data["freq_legal_per_10k"],
    c=plot_data["z"], cmap="coolwarm", s=18, alpha=0.65,
)
plt.xscale("symlog")
plt.yscale("symlog")
plt.xlabel("Consumer frequency per 10k tokens")
plt.ylabel("Legal-formal frequency per 10k tokens")
plt.title("Consumer vs legal-formal keyness scatter")
plt.colorbar(label="log-odds z-score")
for _, row in pd.concat([
    plot_data.sort_values("z", ascending=False).head(8),
    plot_data.sort_values("z").head(8),
]).iterrows():
    plt.annotate(row["term"], (row["freq_consumer_per_10k"], row["freq_legal_per_10k"]), fontsize=8)
plt.tight_layout()
plt.show()
print(f"Saved scatter data: {keyness_scatter_path}")


## 9. Inspect risky subsets

Review long, duplicated, template-like, and legal-heavy samples before deciding exclusion rules.


In [ ]:
review_columns = [
    "Complaint ID", "Product", "Issue", "register_bucket",
    "duplicate_group_size", "family_group_size", "word_count", "narrative_raw"
]

display(
    df.sort_values(["family_group_size", "word_count"], ascending=False)
      [review_columns]
      .head(20)
)

display(
    df.loc[df["very_long"], review_columns]
      .sort_values("word_count", ascending=False)
      .head(10)
)


## 10. Export a derived analysis table

Export only derived data for local analysis. Keep the original CSV unchanged and do not publish the raw narratives in the repository (repo policy: download script + hashes + derived labels + synthetic rewrites only).


In [ ]:
derived_columns = [
    "Complaint ID", "Date received", "Product", "Sub-product", "Issue", "Sub-issue",
    "Company", "State", "narrative_raw", "narrative_core", "narrative_normalized",
    "narrative_hash", "family_signature", "duplicate_group_size", "family_group_size",
    "register_bucket", "register_confidence", "register_evidence",
    "char_count", "word_count", "word_count_noplaceholder", "newline_count",
    "redaction_density", "uppercase_ratio",
    "pronoun_density", "caps_emphasis_count", "expressive_punct_count",
    "parser_backend", "negation_cue", "negated_predicate", "negation_scope_text",
    "negation_focus_entity", "embedded_negation_risk", "contrastive_negation",
    "polarity_flip_risk", "passive_event_verb", "passive_agent_present",
    "agentless_financial_passive", "repair_type", "repair_target_slot",
    "old_value_candidate", "new_value_candidate", "requires_route_recompute",
    "date_mentions_json", "date_role_count", "date_conflict_flag", "temporal_role_summary",
    "speech_act_layers", "stage_order_pattern", "stage_offsets_json",
    "narrative_sequence_type", "nonlinear_narrative_flag", "spanish_anchor_count",
    "non_ascii_latin_count", "language_variation_candidate", "redaction_type_candidates",
    "claim_type_candidate", "card_type_candidate", "payment_rail_candidate", "route_hint",
    "missing_slots", "present_slots", "missing_slot_count", "risk_flags", "risk_flag_count",
    "human_review_reason", "scenario_seed_candidate",
    "negation_count", "hedging_count", "temporal_anchor_count", "repair_candidate_count",
    "implicit_slot_penalty", "complexity_score", "complexity_tier",
] + quality_flags + linguistic_flags + density_columns
derived_columns = list(dict.fromkeys(derived_columns))

derived = df[derived_columns].copy()
local_parquet_path = OUTPUT_DIR / "complaints-2026-06-09_00_45_linguistic_ready_local_v4.parquet"
repo_safe_path = OUTPUT_DIR / "complaints-2026-06-09_00_45_linguistic_features_repo_safe_v4.csv"

# Local full file includes raw narrative for traceability. Do not publish it.
derived.to_parquet(local_parquet_path, index=False)

repo_safe_drop = [
    "narrative_raw", "narrative_core", "narrative_normalized", "negation_scope_text",
    "date_mentions_json", "stage_offsets_json", "scenario_seed_candidate",
]
repo_safe = derived.drop(columns=[col for col in repo_safe_drop if col in derived.columns]).copy()
repo_safe.to_csv(repo_safe_path, index=False, encoding="utf-8-sig")

print(f"Saved local full Parquet: {local_parquet_path}")
print(f"Saved repo-safe CSV:      {repo_safe_path}")


## 11. Stratified annotation sample

Create a reproducible 100-case sample stratified by register. The notebook also creates **rule-assisted candidate labels** for negation, hedging, modality, speech act, emotion/escalation, narrative stages, temporal complexity, and slot realization. Candidate labels support review and error analysis; they are not human annotations or gold labels.

**V3 safeguards**:
- The sampler degrades gracefully when a stratum is short (takes what exists and reports the shortfall) instead of raising, and redistribution is logged so the sampling frame stays auditable.
- Within-stratum product mix is displayed, so a register stratum accidentally dominated by one product (e.g., credit reporting) is visible before annotation begins.
- The blind template contains only text, metadata, and empty `human_*` columns. Make two independent copies for annotators A and B; neither copy exposes candidate labels.
- Candidate labels are exported separately and should remain sealed until both human annotation files are complete.


In [ ]:
SAMPLE_PLAN = {
    "consumer_narrative": 50,
    "legal_formal": 20,
    "pasted_correspondence": 15,
    "template_form": 10,
    "template_letter_family": 5,   # Retain a small mass-template stratum.
}
ANNOTATION_SEED = 20260611

sample_parts = []
shortfalls = {}
for offset, (register, sample_size) in enumerate(SAMPLE_PLAN.items()):
    register_rows = (
        exact_dedup_corpus.loc[exact_dedup_corpus["register_bucket"].eq(register)]
        .drop_duplicates("family_signature", keep="first")
    )
    take = min(sample_size, len(register_rows))
    if take < sample_size:
        shortfalls[register] = sample_size - take
    if take > 0:
        sample_parts.append(
            register_rows.sample(n=take, random_state=ANNOTATION_SEED + offset)
        )

if not sample_parts:
    raise ValueError(
        "No rows matched any stratum in SAMPLE_PLAN. Check register_bucket values: "
        f"{analysis_corpus['register_bucket'].value_counts().to_dict()}"
    )

annotation_sample = pd.concat(sample_parts, ignore_index=True)
target_sample_size = sum(SAMPLE_PLAN.values())
redistribution = {}

# Fill short strata from unused normalization families while preserving the total target.
remaining_needed = target_sample_size - len(annotation_sample)
if remaining_needed > 0:
    selected_ids = set(annotation_sample["Complaint ID"])
    selected_families = set(annotation_sample["family_signature"])
    reserve = (
        exact_dedup_corpus.loc[
            ~exact_dedup_corpus["Complaint ID"].isin(selected_ids)
            & ~exact_dedup_corpus["family_signature"].isin(selected_families)
        ]
        .drop_duplicates("family_signature", keep="first")
    )
    fill_count = min(remaining_needed, len(reserve))
    if fill_count:
        fill_rows = reserve.sample(
            n=fill_count, random_state=ANNOTATION_SEED + 10_000
        )
        redistribution = fill_rows["register_bucket"].value_counts().to_dict()
        annotation_sample = pd.concat(
            [annotation_sample, fill_rows], ignore_index=True
        )

annotation_sample = (
    annotation_sample.sample(frac=1, random_state=ANNOTATION_SEED)
    .reset_index(drop=True)
)
annotation_sample.insert(
    0, "annotation_id",
    [f"CFPB-{i:03d}" for i in range(1, len(annotation_sample) + 1)],
)

if shortfalls:
    print(f"Strata short of initial plan: {shortfalls}")
    print(f"Redistributed replacements: {redistribution}")
if len(annotation_sample) != target_sample_size:
    print(
        f"WARNING: requested {target_sample_size} cases but only "
        f"{len(annotation_sample)} unique-family cases were available."
    )

display(annotation_sample["register_bucket"].value_counts().to_frame("sample_count"))
display(
    pd.crosstab(annotation_sample["register_bucket"], annotation_sample["Product"])
)


In [ ]:
INDIRECT_REQUEST_RE = re.compile(
    r"\b(?:can you|could you|would you|will you|I wonder if|I hope you can|"
    r"why (?:hasn't|haven't|didn't|won't))\b",
    re.I,
)
RHETORICAL_COMPLAINT_RE = re.compile(
    r"\b(?:is this a joke|how (?:can|could) (?:you|they)|what kind of|"
    r"how is this (?:fair|legal|possible))\b",
    re.I,
)
OPENING_RE = re.compile(
    r"\b(?:I am filing|I am submitting|I have an issue|I am writing|this complaint|"
    r"I need to dispute|I want to report)\b",
    re.I,
)
EVENT_RE = re.compile(
    r"\b(?:charged|withdrew|removed|transferred|payment|purchase|transaction|"
    r"account was|card was|reported|collection)\b",
    re.I,
)
ATTRIBUTION_RE = re.compile(
    r"\b(?:because|due to|merchant|company|bank|fraud|scam|stole|stolen|hacked|"
    r"identity theft|made a mistake|error)\b",
    re.I,
)
IMPACT_RE = re.compile(
    r"\b(?:rent|food|medicine|bills?|financial hardship|damag\w* my credit|"
    r"unable to|could not access|caused me|stress|distress)\b",
    re.I,
)
CODA_RE = re.compile(
    r"(?:can you help|please help|thank you|sincerely|I look forward to|"
    r"please advise)[.!?\s]*$",
    re.I,
)
TIME_TOKEN_RE = re.compile(
    r"\[DATE\]|\b(?:yesterday|today|last (?:week|month|year|night)|"
    r"this (?:week|month)|\d+ days? ago|recently|since|before|after)\b",
    re.I,
)
NONLINEAR_TIME_RE = re.compile(
    r"\b(?:but|although|however)\b.{0,100}\b(?:before|after|earlier|later|"
    r"last week|yesterday|\[DATE\])\b",
    re.I | re.S,
)
CARD_TYPE_RE = re.compile(
    r"\b(?:credit card|debit card|prepaid card|Zelle|ACH|wire transfer|checking account|"
    r"savings account|mortgage|student loan|vehicle loan)\b",
    re.I,
)
MERCHANT_RE = re.compile(
    r"\b(?:merchant|retailer|store|restaurant|vendor|Amazon|Walmart|Uber|Costco|"
    r"company|collector|lender|bank)\b",
    re.I,
)

for flag, pattern in LINGUISTIC_PATTERNS.items():
    annotation_sample[flag] = annotation_sample["narrative_normalized"].str.contains(
        pattern, na=False
    )


def label_emotion_escalation(row: pd.Series) -> str:
    emotion = bool(row["lx_emotion"])
    escalation = bool(row["lx_escalation_signal"])
    if emotion and escalation:
        return "emotion_and_escalation"
    if emotion:
        return "emotion_only"
    if escalation:
        return "escalation_only"
    return "none_detected"


def label_modality(row: pd.Series) -> str:
    epistemic = bool(row["lx_modality_epistemic"])
    deontic = bool(row["lx_modality_deontic"])
    if epistemic and deontic:
        return "epistemic_and_deontic"
    if epistemic:
        return "epistemic_only"
    if deontic:
        return "deontic_only"
    return "none_detected"


def label_speech_act_candidate(text: str) -> str:
    if INDIRECT_REQUEST_RE.search(text):
        return "indirect_request_candidate"
    if RHETORICAL_COMPLAINT_RE.search(text):
        return "rhetorical_complaint_candidate"
    if "?" in text:
        return "interrogative_other"
    return "none_detected"


def label_narrative_stages(text: str, has_request: bool) -> str:
    stages = []
    if OPENING_RE.search(text):
        stages.append("opening")
    if EVENT_RE.search(text):
        stages.append("event")
    if ATTRIBUTION_RE.search(text):
        stages.append("attribution")
    if IMPACT_RE.search(text):
        stages.append("impact")
    if has_request:
        stages.append("request")
    if CODA_RE.search(text):
        stages.append("coda")
    return "|".join(stages) if stages else "none_detected"


def label_temporal_complexity(text: str) -> str:
    anchors = TIME_TOKEN_RE.findall(text)
    if not anchors:
        return "no_time_anchor"
    if NONLINEAR_TIME_RE.search(text):
        return "nonlinear_or_contrastive"
    if len(anchors) == 1:
        return "single_time_anchor"
    return "multiple_time_anchors"


def label_slot_realization(text: str) -> tuple[str, str]:
    slots = {
        "amount": bool(re.search(r"\[AMOUNT\]|\$\s*\d", text)),
        "date": bool(re.search(r"\[DATE\]", text)),
        "account_or_card_type": bool(CARD_TYPE_RE.search(text)),
        "merchant_or_institution": bool(MERCHANT_RE.search(text)),
        "claim_signal": bool(LINGUISTIC_PATTERNS["lx_authorization_language"].search(text)),
    }
    explicit_slots = [name for name, present in slots.items() if present]
    if len(explicit_slots) >= 4:
        realization = "explicit_rich"
    elif len(explicit_slots) >= 2:
        realization = "explicit_partial"
    elif len(explicit_slots) == 1:
        realization = "minimal_explicit"
    else:
        realization = "implicit_or_missing"
    return realization, "|".join(explicit_slots) if explicit_slots else "none_detected"


annotation_sample["candidate_negation"] = annotation_sample["lx_negation"].astype(bool)
annotation_sample["candidate_hedging"] = annotation_sample["lx_hedging"].astype(bool)
annotation_sample["candidate_modality"] = annotation_sample.apply(label_modality, axis=1)
annotation_sample["candidate_speech_act"] = annotation_sample["narrative_normalized"].map(
    label_speech_act_candidate
)
annotation_sample["candidate_emotion_escalation"] = annotation_sample.apply(
    label_emotion_escalation, axis=1
)
annotation_sample["candidate_narrative_stages"] = annotation_sample.apply(
    lambda row: label_narrative_stages(
        row["narrative_normalized"], bool(row["lx_request_or_demand"])
    ),
    axis=1,
)
annotation_sample["candidate_temporal_complexity"] = annotation_sample["narrative_normalized"].map(
    label_temporal_complexity
)
slot_labels = annotation_sample["narrative_normalized"].map(label_slot_realization)
annotation_sample["candidate_slot_realization"] = slot_labels.str[0]
annotation_sample["candidate_explicit_slots"] = slot_labels.str[1]

annotation_sample["annotation_source"] = "v4_rule_assisted_candidates"
annotation_sample["candidate_generation_complete"] = True


### 11b. Export blind template and candidate labels

Two files, two purposes:
- `..._blind.csv`: duplicate this template for annotators A and B. It contains text, metadata, and empty `human_*` columns, but no machine candidates.
- `..._candidates.csv`: heuristic candidate labels for later validation and adjudication support.

Both carry the same `annotation_id`, so they re-join trivially.


In [ ]:
HUMAN_LABEL_COLUMNS = [
    "human_negation", "human_hedging", "human_modality", "human_speech_act",
    "human_emotion_escalation", "human_narrative_stages", "human_temporal_complexity",
    "human_slot_realization", "human_claim_type", "human_payment_rail",
    "human_route_hint", "human_missing_slots", "human_risk_flags", "human_notes",
]

shared_columns = [
    "annotation_id", "Complaint ID", "Product", "Sub-product", "Issue", "Sub-issue",
    "register_bucket", "word_count", "narrative_raw", "narrative_normalized",
]
machine_columns = [
    "candidate_negation", "candidate_hedging", "candidate_modality", "candidate_speech_act",
    "candidate_emotion_escalation", "candidate_narrative_stages", "candidate_temporal_complexity",
    "candidate_slot_realization", "candidate_explicit_slots", "claim_type_candidate",
    "card_type_candidate", "payment_rail_candidate", "route_hint", "missing_slots",
    "risk_flags", "human_review_reason", "complexity_score", "complexity_tier",
    "speech_act_layers", "negation_scope_text", "passive_event_verb", "repair_target_slot",
    "temporal_role_summary", "stage_order_pattern", "annotation_source",
    "candidate_generation_complete",
]

# Blind file: no machine labels, empty human columns to fill.
blind = annotation_sample[shared_columns].copy()
for column in HUMAN_LABEL_COLUMNS:
    blind[column] = pd.NA
blind_path = OUTPUT_DIR / "cfpb_linguistic_annotation_sample_blind_v4.csv"
blind.to_csv(blind_path, index=False, encoding="utf-8-sig")

# Candidate file: keep sealed until independent human annotation is complete.
candidates = annotation_sample[shared_columns + machine_columns].copy()
candidates_path = OUTPUT_DIR / "cfpb_linguistic_annotation_candidates_v4.csv"
candidates.to_csv(candidates_path, index=False, encoding="utf-8-sig")

annotation_summary = pd.DataFrame({
    "negation": annotation_sample["candidate_negation"].value_counts(),
    "hedging": annotation_sample["candidate_hedging"].value_counts(),
}).fillna(0).astype(int)

display(annotation_summary)
display(annotation_sample["candidate_modality"].value_counts().to_frame("count"))
display(annotation_sample["candidate_speech_act"].value_counts().to_frame("count"))
display(annotation_sample["candidate_emotion_escalation"].value_counts().to_frame("count"))
display(annotation_sample["candidate_temporal_complexity"].value_counts().to_frame("count"))
display(annotation_sample["candidate_slot_realization"].value_counts().to_frame("count"))

print(f"Saved blind file:      {blind_path}")
print(f"Saved candidate file:  {candidates_path}")


### 11c. Phenomenon-stratified stress sample

The register-stratified sample estimates corpus variety. FinDisputeEval also needs high-risk linguistic phenomena for evaluation. V4 therefore exports a second sample that deliberately over-represents negation scope, repair, temporal complexity, authorization ambiguity, route uncertainty, language variation, and product/claim edge cases. This sample is for stress testing, not prevalence estimation.


In [ ]:
analysis_corpus = add_complexity_features(analysis_corpus)

PHENOMENON_SAMPLE_PLAN = {
    "negation_heavy": 8,
    "contrastive_or_embedded_negation": 8,
    "hedging_epistemic_uncertainty": 8,
    "deontic_demand": 8,
    "indirect_or_rhetorical_speech_act": 8,
    "authorization_ambiguity": 8,
    "temporal_complexity": 8,
    "nonlinear_narrative": 8,
    "escalation_or_legal_threat": 8,
    "implicit_or_missing_slots": 8,
    "language_variation_candidate": 8,
    "zelle_ach_wire_or_cnp": 8,
    "duplicate_or_recurring_claim": 8,
    "third_party_or_family_auth": 8,
    "template_legal_controls": 8,
}
phenomenon_masks = {
    "negation_heavy": analysis_corpus["negation_count"].ge(max(2, analysis_corpus["negation_count"].quantile(0.85))),
    "contrastive_or_embedded_negation": analysis_corpus["contrastive_negation"] | analysis_corpus["embedded_negation_risk"],
    "hedging_epistemic_uncertainty": analysis_corpus["lx_hedging"] | analysis_corpus["lx_modality_epistemic"],
    "deontic_demand": analysis_corpus["lx_modality_deontic"] | analysis_corpus["speech_act_layers"].str.contains("demand_action", na=False),
    "indirect_or_rhetorical_speech_act": analysis_corpus["speech_act_layers"].str.contains("indirect_request|rhetorical_complaint", na=False),
    "authorization_ambiguity": analysis_corpus["polarity_flip_risk"] | analysis_corpus["claim_type_candidate"].eq("unauthorized"),
    "temporal_complexity": analysis_corpus["date_role_count"].ge(2) | analysis_corpus["temporal_anchor_count"].ge(2),
    "nonlinear_narrative": analysis_corpus["nonlinear_narrative_flag"],
    "escalation_or_legal_threat": analysis_corpus["risk_flags"].str.contains("legal_threat|regulator_mention|bank_refused_escalation", na=False),
    "implicit_or_missing_slots": analysis_corpus["missing_slot_count"].ge(6),
    "language_variation_candidate": analysis_corpus["language_variation_candidate"],
    "zelle_ach_wire_or_cnp": analysis_corpus["payment_rail_candidate"].isin(["zelle", "ach", "wire", "card_not_present"]),
    "duplicate_or_recurring_claim": analysis_corpus["claim_type_candidate"].isin(["duplicate", "subscription_recurring"]),
    "third_party_or_family_auth": analysis_corpus["present_slots"].str.contains("third_party_auth", na=False),
    "template_legal_controls": analysis_corpus["register_bucket"].isin(["legal_formal", "template_form", "template_letter_family"]),
}

selected_ids = set()
selected_families = set()
stress_parts = []
stress_shortfalls = {}
for offset, (stratum, target_n) in enumerate(PHENOMENON_SAMPLE_PLAN.items()):
    pool = analysis_corpus.loc[phenomenon_masks[stratum]].copy()
    pool = pool.loc[
        ~pool["Complaint ID"].isin(selected_ids)
        & ~pool["family_signature"].isin(selected_families)
    ]
    take = min(target_n, len(pool))
    if take < target_n:
        stress_shortfalls[stratum] = target_n - take
    if take:
        sample = pool.sample(n=take, random_state=ANNOTATION_SEED + 20_000 + offset).copy()
        sample["phenomenon_stratum"] = stratum
        stress_parts.append(sample)
        selected_ids.update(sample["Complaint ID"])
        selected_families.update(sample["family_signature"])

stress_sample = pd.concat(stress_parts, ignore_index=True) if stress_parts else analysis_corpus.head(0).copy()
stress_target = sum(PHENOMENON_SAMPLE_PLAN.values())
stress_redistribution = {}
remaining_needed = stress_target - len(stress_sample)
if remaining_needed > 0:
    reserve = analysis_corpus.loc[
        ~analysis_corpus["Complaint ID"].isin(selected_ids)
        & ~analysis_corpus["family_signature"].isin(selected_families)
    ].sort_values("complexity_score", ascending=False)
    fill = reserve.head(remaining_needed).copy()
    if len(fill):
        fill["phenomenon_stratum"] = "complexity_backfill"
        stress_redistribution = fill["complexity_tier"].astype(str).value_counts().to_dict()
        stress_sample = pd.concat([stress_sample, fill], ignore_index=True)

stress_sample = stress_sample.sample(frac=1, random_state=ANNOTATION_SEED + 30_000).reset_index(drop=True)
stress_sample.insert(0, "stress_sample_id", [f"STRESS-{i:03d}" for i in range(1, len(stress_sample) + 1)])
stress_columns = [
    "stress_sample_id", "phenomenon_stratum", "Complaint ID", "Product", "Issue", "register_bucket",
    "complexity_score", "complexity_tier", "claim_type_candidate", "payment_rail_candidate",
    "route_hint", "missing_slots", "risk_flags", "negation_scope_text", "speech_act_layers",
    "temporal_role_summary", "stage_order_pattern", "narrative_normalized",
]
stress_sample_path = OUTPUT_DIR / "cfpb_phenomenon_stress_sample_v4.csv"
stress_sample[stress_columns].to_csv(stress_sample_path, index=False, encoding="utf-8-sig")
print(f"Phenomenon stress rows: {len(stress_sample):,} / requested {stress_target:,}")
if stress_shortfalls:
    print("Shortfalls:", stress_shortfalls)
if stress_redistribution:
    print("Complexity backfill:", stress_redistribution)
display(stress_sample["phenomenon_stratum"].value_counts().to_frame("sample_count"))
display(stress_sample["complexity_tier"].value_counts().to_frame("sample_count"))
print(f"Saved: {stress_sample_path}")


### 11d. Synthetic-data parameter extraction

This JSON is the handoff from corpus EDA to synthetic generation. It captures register distribution, phenomenon rates, claim/payment routing hypotheses, missing-slot distribution, and safety constraints. It is intended as input to a generator/validator pipeline, not as a claim that CFPB narratives are complete banking chats.


In [ ]:
risk_counter = count_pipe_values(analysis_corpus["risk_flags"])
missing_slot_counter = count_pipe_values(analysis_corpus["missing_slots"])
synthetic_params = {
    "source_corpus": CORPUS_METADATA,
    "notebook_version": NOTEBOOK_VERSION,
    "parser_status": PARSER_STATUS,
    "corpus_boundaries": {
        "cfpb_published_narratives_only": True,
        "not_real_time_chat": True,
        "not_legal_ground_truth": True,
        "use_as": ["real-language anchors", "scenario seeds", "handoff benchmark", "synthetic distribution constraints"],
    },
    "register_distribution": analysis_corpus["register_bucket"].value_counts(normalize=True).round(6).to_dict(),
    "claim_type_distribution": analysis_corpus["claim_type_candidate"].value_counts(normalize=True).round(6).to_dict(),
    "payment_rail_distribution": analysis_corpus["payment_rail_candidate"].value_counts(normalize=True).round(6).to_dict(),
    "route_hint_distribution": analysis_corpus["route_hint"].value_counts(normalize=True).round(6).to_dict(),
    "phenomenon_targets": {
        "negation_density_median": float(analysis_corpus["lx_negation_density"].median()),
        "hedging_rate": float(analysis_corpus["lx_hedging"].mean()),
        "temporal_expression_rate": float(analysis_corpus["lx_temporal_expression"].mean()),
        "authorization_language_rate": float(analysis_corpus["lx_authorization_language"].mean()),
        "agentless_financial_passive_rate": float(analysis_corpus["agentless_financial_passive"].mean()),
        "repair_candidate_rate": float(analysis_corpus["repair_type"].ne("none_detected").mean()),
        "polarity_flip_risk_rate": float(analysis_corpus["polarity_flip_risk"].mean()),
        "nonlinear_narrative_rate": float(analysis_corpus["nonlinear_narrative_flag"].mean()),
        "language_variation_candidate_rate": float(analysis_corpus["language_variation_candidate"].mean()),
    },
    "missing_slot_rates": {slot: round(count / len(analysis_corpus), 6) for slot, count in missing_slot_counter.items()},
    "risk_flag_rates": {flag: round(count / len(analysis_corpus), 6) for flag, count in risk_counter.items()},
    "complexity_distribution": analysis_corpus["complexity_tier"].astype(str).value_counts(normalize=True).round(6).to_dict(),
    "seed_generation_constraints": {
        "must_include_missing_slots": True,
        "must_include_route_hint": True,
        "must_distinguish_real_vs_synthetic": True,
        "must_not_train_final_legal_decision_model_from_cfpb_labels": True,
        "must_preserve_negation_modality_pronouns_punctuation_case_for_language_analysis": True,
    },
    "recommended_alignment_checks": [
        "register_distribution", "phenomenon_rate_density", "claim_type_distribution",
        "missing_slot_distribution", "risk_flag_distribution", "tfidf_keyness_drift",
        "embedding_distance", "jensen_shannon_divergence", "self_bleu", "rare_edge_case_coverage",
    ],
    "language_detector_coverage": language_coverage_note,
    "artifacts": {
        "scenario_seed_jsonl": scenario_seed_path.name,
        "phenomenon_stress_sample": stress_sample_path.name,
    },
}
synthetic_params_path = OUTPUT_DIR / "cfpb_synthetic_generation_params_v4.json"
synthetic_params_path.write_text(json.dumps(synthetic_params, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(synthetic_params, indent=2, ensure_ascii=False)[:4000])
print(f"Saved: {synthetic_params_path}")


### 11e. Annotation guideline starter

V4 writes a lightweight guideline starter so the annotation loop has a versioned document tied to the same artifacts. Treat it as a first draft for human adjudication, not a finished codebook.


In [ ]:
annotation_guidelines = f"""# FinDisputeEval CFPB Annotation Guidelines v4

Scope: annotate linguistic and scenario-seed candidates from CFPB published narratives. These are intake hypotheses, not legal findings.

## Files
- Blind template: `{blind_path.name}`. Duplicate into annotator A/B copies before labeling.
- Candidate labels: `{candidates_path.name}`. Keep sealed until both blind files are complete.
- Phenomenon stress sample: `{stress_sample_path.name}`. Use for guideline refinement and low-base-rate phenomena.

## Required principles
1. Preserve negation, modality, pronouns, punctuation, and casing as linguistic evidence.
2. Do not infer a final legal outcome from a CFPB narrative.
3. Mark uncertainty explicitly instead of forcing a label.
4. Record adjudicated labels in separate columns; never overwrite annotator A/B labels.
5. For low prevalence labels, report observed agreement, positive agreement, negative agreement, Cohen's kappa, and PABAK.

## Core labels
- `human_claim_type`: unauthorized, duplicate, non_delivery, subscription_recurring, merchant_dispute, billing_error, fee_interest_dispute, debt_validation, credit_reporting_dispute, identity_theft_fraud, unknown_ambiguous.
- `human_payment_rail`: debit_card, credit_card, prepaid_card, checking_account, savings_account, zelle, ach, wire, card_not_present, unknown.
- `human_route_hint`: non-final regulatory route hypothesis; use unknown when card/payment rail is missing.
- `human_missing_slots`: pipe-separated missing intake slots.
- `human_risk_flags`: pipe-separated risk flags.

## Notes
Use the stress sample to refine edge-case guidance before producing adjudicated gold labels.
"""
annotation_guidelines_path = OUTPUT_DIR / "annotation_guidelines_v4.md"
annotation_guidelines_path.write_text(annotation_guidelines, encoding="utf-8")
print(f"Saved: {annotation_guidelines_path}")


### 11f. Sampling manifest (lineage sidecar)

A JSON manifest binds the samples, seeds, parser status, corpus snapshot, seeds, shortfalls, and detector inventory. Anyone re-running the notebook against the same CSV should reproduce the same annotation mappings; if not, the manifest tells you where drift occurred.


In [ ]:
pattern_inventory_hash = hashlib.sha1(
    json.dumps(
        {name: pattern.pattern for name, pattern in LINGUISTIC_PATTERNS.items()},
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "corpus": CORPUS_METADATA,
    "annotation_seed": ANNOTATION_SEED,
    "sample_plan": SAMPLE_PLAN,
    "sample_shortfalls": shortfalls,
    "sample_redistribution": redistribution,
    "phenomenon_sample_plan": PHENOMENON_SAMPLE_PLAN,
    "phenomenon_sample_shortfalls": stress_shortfalls,
    "phenomenon_sample_redistribution": stress_redistribution,
    "phenomenon_sample_size": int(len(stress_sample)),
    "sample_size": int(len(annotation_sample)),
    "parser_status": PARSER_STATUS,
    "corpus_view_sizes": {
        "all": int(len(all_corpus)),
        "exact_dedup": int(len(exact_dedup_corpus)),
        "family_dedup": int(len(family_dedup_corpus)),
    },
    "linguistic_pattern_inventory_sha1": pattern_inventory_hash,
    "outputs": {
        "blind_file": blind_path.name,
        "candidates_file": candidates_path.name,
        "pattern_audit_file": pattern_audit_path.name,
        "register_audit_file": register_audit_path.name,
        "scenario_seed_jsonl": scenario_seed_path.name,
        "phenomenon_stress_sample": stress_sample_path.name,
        "synthetic_params_file": synthetic_params_path.name,
        "annotation_guidelines_file": annotation_guidelines_path.name,
        "local_full_parquet": local_parquet_path.name,
        "repo_safe_features_csv": repo_safe_path.name,
        "collocations_file": collocation_path.name,
        "keyness_scatter_file": keyness_scatter_path.name,
    },
    "id_mapping_sha1": hashlib.sha1(
        annotation_sample[["annotation_id", "Complaint ID"]]
        .to_csv(index=False).encode("utf-8")
    ).hexdigest(),
}

manifest_path = OUTPUT_DIR / "cfpb_annotation_sample_manifest_v4.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print(f"Saved: {manifest_path}")


## 12. Human IAA and heuristic validation

After two annotators independently complete copies of the blind template, save them as `cfpb_annotations_annotator_a_completed_v4.csv` and `cfpb_annotations_annotator_b_completed_v4.csv`. This cell computes:
- **Cohen's kappa** and observed agreement for single-label fields.
- **Mean Jaccard similarity** for the multi-label narrative-stage field.
- Candidate-vs-human precision, recall, and F1 separately for supported binary fields. This is heuristic validation, not IAA.

The cell is a no-op with a clear message until both completed annotation files exist.


In [ ]:
def paired_nonempty(left: pd.Series, right: pd.Series) -> pd.DataFrame:
    paired = pd.DataFrame({"left": left, "right": right})
    paired = paired.replace(r"^\s*$", pd.NA, regex=True).dropna()
    return paired


def binary_agreement_details(left: pd.Series, right: pd.Series) -> dict:
    paired = paired_nonempty(left, right)
    if paired.empty:
        return {"positive_agreement": np.nan, "negative_agreement": np.nan, "pabak": np.nan}
    l = paired["left"].astype("string").str.lower().map({"true": True, "false": False})
    r = paired["right"].astype("string").str.lower().map({"true": True, "false": False})
    valid = l.notna() & r.notna()
    l = l[valid]
    r = r[valid]
    if len(l) == 0:
        return {"positive_agreement": np.nan, "negative_agreement": np.nan, "pabak": np.nan}
    tp = int((l & r).sum())
    tn = int((~l & ~r).sum())
    fp = int((l & ~r).sum())
    fn = int((~l & r).sum())
    observed = (tp + tn) / max(1, tp + tn + fp + fn)
    positive_agreement = 2 * tp / max(1, 2 * tp + fp + fn)
    negative_agreement = 2 * tn / max(1, 2 * tn + fp + fn)
    return {
        "positive_agreement": positive_agreement,
        "negative_agreement": negative_agreement,
        "pabak": 2 * observed - 1,
    }


def multilabel_jaccard(left: pd.Series, right: pd.Series) -> float:
    paired = paired_nonempty(left, right)
    if paired.empty:
        return float("nan")
    scores = []
    for _, row in paired.iterrows():
        a = {value for value in str(row["left"]).split("|") if value}
        b = {value for value in str(row["right"]).split("|") if value}
        scores.append(len(a & b) / len(a | b) if (a | b) else 1.0)
    return float(np.mean(scores))


annotator_a_path = OUTPUT_DIR / "cfpb_annotations_annotator_a_completed_v4.csv"
annotator_b_path = OUTPUT_DIR / "cfpb_annotations_annotator_b_completed_v4.csv"

if annotator_a_path.exists() and annotator_b_path.exists():
    annotator_a = pd.read_csv(annotator_a_path, dtype={"annotation_id": "string"})
    annotator_b = pd.read_csv(annotator_b_path, dtype={"annotation_id": "string"})
    human_pair = annotator_a.merge(
        annotator_b, on="annotation_id", suffixes=("_a", "_b"), validate="one_to_one"
    )
    single_label_fields = [
        "human_negation", "human_hedging", "human_modality", "human_speech_act",
        "human_emotion_escalation", "human_temporal_complexity", "human_slot_realization",
    ]
    iaa_rows = []
    for field in single_label_fields:
        paired = paired_nonempty(human_pair[f"{field}_a"], human_pair[f"{field}_b"])
        iaa_rows.append({
            "phenomenon": field.removeprefix("human_"),
            "n": len(paired),
            "observed_agreement": (paired["left"] == paired["right"]).mean() if len(paired) else np.nan,
            "cohens_kappa": cohen_kappa_score(paired["left"], paired["right"]) if len(paired) else np.nan,
            **binary_agreement_details(human_pair[f"{field}_a"], human_pair[f"{field}_b"]),
        })
    iaa = pd.DataFrame(iaa_rows)
    narrative_jaccard = multilabel_jaccard(
        human_pair["human_narrative_stages_a"], human_pair["human_narrative_stages_b"]
    )
    display(iaa.style.format({"observed_agreement": "{:.3f}", "cohens_kappa": "{:.3f}", "positive_agreement": "{:.3f}", "negative_agreement": "{:.3f}", "pabak": "{:.3f}"}))
    print(f"Narrative-stage mean Jaccard: {narrative_jaccard:.3f}")
else:
    print(
        "True IAA requires two independently completed files:\n"
        f"- {annotator_a_path.name}\n- {annotator_b_path.name}"
    )

# Candidate-vs-human validation is reported separately and is not IAA.
if annotator_a_path.exists():
    annotator_a = pd.read_csv(annotator_a_path, dtype={"annotation_id": "string"})
    validation = candidates.merge(
        annotator_a[["annotation_id"] + HUMAN_LABEL_COLUMNS],
        on="annotation_id", how="inner", validate="one_to_one",
    )
    binary_pairs = [
        ("candidate_negation", "human_negation"),
        ("candidate_hedging", "human_hedging"),
    ]
    validation_rows = []
    for candidate_col, human_col in binary_pairs:
        paired = paired_nonempty(validation[candidate_col], validation[human_col])
        if paired.empty:
            continue
        truth = paired["right"].astype("string").str.lower().map({"true": True, "false": False})
        pred = paired["left"].astype("string").str.lower().map({"true": True, "false": False})
        valid = truth.notna() & pred.notna()
        precision, recall, f1, _ = precision_recall_fscore_support(
            truth[valid], pred[valid], average="binary", zero_division=0
        )
        validation_rows.append({
            "phenomenon": candidate_col.removeprefix("candidate_"),
            "n": int(valid.sum()), "precision": precision, "recall": recall, "f1": f1,
        })
    display(pd.DataFrame(validation_rows).style.format({
        "precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}"
    }))


## Annotation boundary

The stratified sample, blind template, candidate-label file, precision-audit file, and manifest have now been generated. Before using any of this as a gold dataset:

1. Create two copies of the **blind** template. Annotators A and B fill the `human_*` columns independently while the candidate file remains sealed.
2. Inspect per-phenomenon agreement (§12). Low-agreement phenomena return to guideline revision before adjudication.
3. Resolve disagreements using the project annotation decision tree; record adjudicated labels in a third column set, never by overwriting either source.
4. Complete the pattern precision audit (§7b) before quoting any regex prevalence in documentation or interviews.
5. Keep regex/model-assisted labels distinct from adjudicated gold labels everywhere downstream.

**Known limits of this notebook (deliberate, documented):**
- Lexical negation density ≠ negation *scope*. Scope/focus resolution needs syntactic structure (e.g., spaCy dependency parses) and is the next analysis layer, not a regex problem.
- The code-switching detector covers a small set of Spanish anchors only; it estimates a lower bound and says nothing about other languages or intra-word switching.
- Narrative-stage labels approximate Labov structure with surface cues; stage *order* is not validated, only stage presence.
